# SDP Analysis Results (all four ablation modes)

This notebook generates the tables and figures of the paper from the pipeline outputs. Nothing is hard-coded: imbalance ratios, feature-reduction percentages, mixed-model coefficients, Pareto fronts and parameter counts are all extracted from the result folders.

**Inputs** (written by `revised_comprehensive_software_defects.py`):

```
<RESULTS_ROOT>/results_no_smote/                     Mode 1: No-SMOTE, No-FS
<RESULTS_ROOT>/results_smote/                        Mode 2: SMOTE,    No-FS
<RESULTS_ROOT>/results_no_smote_feature_selection/   Mode 3: No-SMOTE, FS
<RESULTS_ROOT>/results_smote_feature_selection/      Mode 4: SMOTE,    FS
    ALL_DATASETS_foldwise_metrics.csv
    <DATASET>/ (or <DATASET>.zip)
        <Model>/fold<k>_predictions.csv
        feature_selection/fold_<k>/feature_importance.csv   (Modes 3-4)
```

**Outputs** go to `<RESULTS_ROOT>/paper_outputs_revision/` in numbered sub-folders (tables as `.csv` and `.tex`, figures as `.png` and `.pdf`), plus a zip of all outputs.

| Section | Content |
|---|---|
| 0. Data checks | Analysis population (observation counts), dataset characteristics, optional raw-data audit (duplicates, conflicting labels) |
| 1. Base learners | Performance table, ROC-AUC heatmap and family boxplot for each of the four modes |
| 2. Feature-selection analysis | RF vs SHAP agreement, cross-fold stability (RF-only, SHAP-only, consensus), percentile-threshold sensitivity, selection frequency, selection overhead |
| 3. Feature-selection and SMOTE effects | Overall, conditional, per-family, per-model and per-dataset effects; paired tests; FS × SMOTE interaction contrast |
| 4. Statistics | Family mixed-effects models with Holm-adjusted contrasts, residual diagnostics, paired dataset-level tests, Friedman/Nemenyi, Family × SMOTE × FS interaction model, forest plot |
| 5. Ensembles | Composition, comparison with the best base learner, cost accounting (total, base, incremental) |
| 6. Computational efficiency | Training/inference cost, latency per 1,000 predictions, net GPU memory, Pareto front, parameter counts |
| 7. Export | Zip of all outputs |

In [ ]:
!pip -q install adjustText scikit-posthocs statsmodels

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Colab -- skipping Google Drive mount.")

In [ ]:
# ==========================================================================
# CONFIGURATION  -- the only cell you normally need to edit
# ==========================================================================
import os, io, re, json, glob, zipfile, itertools, warnings, textwrap, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import studentized_range
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

# Root folder that holds the four ablation result folders written by the
# REVISED pipeline (OUTPUT_ROOT in revised_comprehensive_software_defects.py).
RESULTS_ROOT = os.environ.get("SDP_RESULTS_ROOT",
                              "/content/drive/MyDrive/Software_Defects_Revised")

# Where every table / figure of this notebook is written.
OUT_ROOT = os.environ.get("SDP_OUT_ROOT",
                          os.path.join(RESULTS_ROOT, "paper_outputs_revision"))

# OPTIONAL: folder with the raw .arff/.csv datasets (same as DATA_DIR in the
# pipeline). Only used for the duplicate / conflicting-label audit.
# If it does not exist the audit is skipped and everything else still runs.
RAW_DATA_DIR = os.environ.get("SDP_RAW_DATA_DIR", "/content/defect_data")

MODE_FOLDERS = {            # mode: (folder, SMOTE, FS)
    1: ("results_no_smote",                   False, False),
    2: ("results_smote",                       True, False),
    3: ("results_no_smote_feature_selection", False,  True),
    4: ("results_smote_feature_selection",     True,  True),
}
MODE_LABELS = {1: "Mode 1: No-SMOTE / No-FS", 2: "Mode 2: SMOTE / No-FS",
               3: "Mode 3: No-SMOTE / FS",    4: "Mode 4: SMOTE / FS"}
MODE_SHORT = {1: "M1", 2: "M2", 3: "M3", 4: "M4"}
FOLD_FILE_NAME = "ALL_DATASETS_foldwise_metrics.csv"

REFERENCE_MODE = 4          # "final pipeline" used for ensemble / efficiency tables
N_FOLDS = 5
META_INNER_SPLITS = 3       # must equal META_INNER_SPLITS in the pipeline
FS_PERCENTILE_USED = 50     # THRESHOLD used by select_features_fold()
THRESHOLD_GRID = [25, 33, 40, 50, 60, 67, 75]   # threshold sensitivity analysis

ALPHA = 0.05
PRACTICAL_DELTA_AUC = 0.01  # smallest ROC-AUC difference treated as practically relevant (1 pp)
N_BOOT = 5000
SEED = 42
PRIMARY_METRIC = "ROC_AUC"

# Section 6b: counts neural-model parameters (needs pytorch_tabular)
COMPUTE_PARAM_COUNTS = os.environ.get("SDP_COMPUTE_PARAMS", "1") == "1"

# --------------------------------------------------------------------------
# Model metadata (definitions, not results)
# --------------------------------------------------------------------------
CLASSICAL = ["LogisticRegression", "NaiveBayes", "RandomForest", "ExtraTrees",
             "XGBoost", "LightGBM", "CatBoost"]
DEEP_TABULAR = ["TabNet", "GANDALF", "DANet"]
TRANSFORMER = ["FTTransformer", "TabTransformer", "AutoInt"]
BASE_LEARNERS = CLASSICAL + DEEP_TABULAR + TRANSFORMER
ENSEMBLES = ["Soft Voting", "Hard Voting", "Stacking (sklearn)", "Meta-Stacking"]
ALL_MODELS = BASE_LEARNERS + ENSEMBLES

MODEL_FAMILY = {**{m: "Classical" for m in CLASSICAL},
                **{m: "DeepTabular" for m in DEEP_TABULAR},
                **{m: "Transformer" for m in TRANSFORMER},
                **{m: "Ensemble" for m in ENSEMBLES}}
FAMILY_ORDER = ["Classical", "DeepTabular", "Transformer"]
FAMILY_LABELS = {"Classical": "Classical ML", "DeepTabular": "Deep Tabular",
                 "Transformer": "Transformer", "Ensemble": "Ensemble"}
FAMILY_COLORS = {"Classical": "#4C72B0", "DeepTabular": "#DD8452",
                 "Transformer": "#55A868", "Ensemble": "#8172B3"}

MODEL_DISPLAY = {
    "LogisticRegression": "LR", "NaiveBayes": "NB", "RandomForest": "RF",
    "ExtraTrees": "ET", "XGBoost": "XGBoost", "LightGBM": "LightGBM",
    "CatBoost": "CatBoost", "TabNet": "TabNet", "GANDALF": "GANDALF",
    "DANet": "DANet", "FTTransformer": "FT-Transformer",
    "TabTransformer": "TabTransformer", "AutoInt": "AutoInt",
    "Soft Voting": "Soft Voting", "Hard Voting": "Hard Voting",
    "Stacking (sklearn)": "Stacking", "Meta-Stacking": "Meta-Stacking",
}
# Sub-folder names used by the pipeline's save_fold_artifacts()
MODEL_DIRS = {**{m: m for m in BASE_LEARNERS},
              "Soft Voting": "SoftVoting", "Hard Voting": "HardVoting",
              "Stacking (sklearn)": "Stacking_sklearn",
              "Meta-Stacking": "MetaStacking_All"}

NASA_DATASETS = ["CM1", "JM1", "KC1", "KC3", "MC1", "MC2", "MW1",
                 "PC1", "PC2", "PC3", "PC4", "PC5"]
AEEEM_DATASETS = ["EQ", "JDT", "LC", "ML", "PDE"]
DATASET_REPO = {**{d: "NASA" for d in NASA_DATASETS},
                **{d: "AEEEM" for d in AEEEM_DATASETS}}

METRICS = ["Accuracy", "Precision", "Recall", "F1", "MCC", "ROC_AUC"]
COST_COLS = ["train_time_sec", "infer_time_sec", "train_gpu_mb", "infer_gpu_mb"]
METRIC_DISPLAY = {"Accuracy": "Accuracy", "Precision": "Precision",
                  "Recall": "Recall", "F1": "F1", "MCC": "MCC",
                  "ROC_AUC": "ROC-AUC"}

# --------------------------------------------------------------------------
# Output helpers
# --------------------------------------------------------------------------
SECTIONS = {
    "checks": "00_data_checks", "table1": "01_table1_per_mode",
    "fs": "02_feature_selection", "effects": "03_fs_smote_effects",
    "stats": "04_statistics", "ens": "05_ensembles",
    "comp": "06_computational",
}
for _s in SECTIONS.values():
    os.makedirs(os.path.join(OUT_ROOT, _s), exist_ok=True)


def outpath(section, fname):
    return os.path.join(OUT_ROOT, SECTIONS[section], fname)


def save_table(df, section, name, caption=None, label=None, index=False,
               float_format="%.4f"):
    """Save a table as CSV and as LaTeX (booktabs)."""
    df.to_csv(outpath(section, f"{name}.csv"), index=index)
    try:
        tex = df.to_latex(index=index, float_format=float_format,
                          caption=caption, label=label, escape=True)
        with open(outpath(section, f"{name}.tex"), "w") as f:
            f.write(tex)
    except Exception as e:                       # jinja2 missing etc.
        print(f"  (LaTeX export skipped for {name}: {e})")
    return df


def save_fig(fig, section, name, dpi=300):
    fig.savefig(outpath(section, f"{name}.png"), dpi=dpi, bbox_inches="tight",
                pad_inches=0.02)
    fig.savefig(outpath(section, f"{name}.pdf"), bbox_inches="tight",
                pad_inches=0.02)
    plt.close(fig)


def fmt_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return "n/a"
    return "<0.001" if p < 0.001 else f"{p:.3f}"


def disp(m):
    return MODEL_DISPLAY.get(m, m)


plt.rcParams.update({"font.size": 10, "figure.dpi": 110})
print("RESULTS_ROOT:", RESULTS_ROOT)
print("OUT_ROOT    :", OUT_ROOT)

In [ ]:
# ==========================================================================
# HELPERS: result-folder reader + statistics
# ==========================================================================

class ResultStore:
    """Reads files of one dataset in one mode, from the folder or, if the
    folder was removed, from <DATASET>.zip written by the pipeline."""

    def __init__(self, root):
        self.root = root
        self._zips = {}

    def mode_dir(self, mode):
        return os.path.join(self.root, MODE_FOLDERS[mode][0])

    def _zip(self, mode, ds):
        key = (mode, ds)
        if key not in self._zips:
            zp = os.path.join(self.mode_dir(mode), f"{ds}.zip")
            self._zips[key] = zipfile.ZipFile(zp) if os.path.exists(zp) else None
        return self._zips[key]

    def read_csv(self, mode, ds, rel):
        p = os.path.join(self.mode_dir(mode), ds, rel)
        if os.path.exists(p):
            return pd.read_csv(p)
        z = self._zip(mode, ds)
        if z is not None:
            rel_n = "/" + rel.replace(os.sep, "/")
            hits = [n for n in z.namelist() if ("/" + n).endswith(rel_n)]
            if hits:
                with z.open(hits[0]) as f:
                    return pd.read_csv(f)
        return None

    def list_rel(self, mode, ds, pattern):
        """Relative paths (inside the dataset folder/zip) matching a regex."""
        rx = re.compile(pattern)
        out = set()
        base = os.path.join(self.mode_dir(mode), ds)
        if os.path.isdir(base):
            for r, _, files in os.walk(base):
                for fn in files:
                    rel = os.path.relpath(os.path.join(r, fn), base).replace(os.sep, "/")
                    if rx.search(rel):
                        out.add(rel)
        z = self._zip(mode, ds)
        if z is not None:
            for n in z.namelist():
                n2 = n.lstrip("./")
                if rx.search(n2):
                    out.add(n2)
        return sorted(out)


STORE = ResultStore(RESULTS_ROOT)


# ---------------- statistics ----------------
def holm_adjust(p):
    p = np.asarray(p, float)
    out = np.full_like(p, np.nan)
    ok = ~np.isnan(p)
    if ok.sum():
        out[ok] = multipletests(p[ok], method="holm")[1]
    return out


def cliffs_delta(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    x = x[~np.isnan(x)]; y = y[~np.isnan(y)]
    if len(x) == 0 or len(y) == 0:
        return np.nan
    gt = (x[:, None] > y[None, :]).sum()
    lt = (x[:, None] < y[None, :]).sum()
    return (gt - lt) / (len(x) * len(y))


def cliffs_magnitude(d):
    """Romano et al. (2006) thresholds."""
    if d is None or np.isnan(d):
        return "n/a"
    a = abs(d)
    return ("negligible" if a < 0.147 else "small" if a < 0.33
            else "medium" if a < 0.474 else "large")


def paired_compare(x, y, n_boot=N_BOOT, seed=SEED, tol=1e-9):
    """Paired comparison of two aligned arrays (e.g. per-dataset means).
    Wilcoxon signed-rank + bootstrap CI of the mean difference + effect sizes."""
    x = np.asarray(x, float); y = np.asarray(y, float)
    ok = ~(np.isnan(x) | np.isnan(y))
    x, y = x[ok], y[ok]
    d = x - y
    n = len(d)
    res = dict(n=n, mean_a=np.nan, mean_b=np.nan, mean_diff=np.nan,
               ci_low=np.nan, ci_high=np.nan, median_diff=np.nan,
               wins=np.nan, ties=np.nan, losses=np.nan,
               wilcoxon_W=np.nan, p=np.nan, rank_biserial=np.nan,
               cohen_dz=np.nan, cliffs_delta=np.nan, cliffs_mag="n/a")
    if n == 0:
        return res
    rng = np.random.default_rng(seed)
    boot = d[rng.integers(0, n, (n_boot, n))].mean(axis=1)
    res.update(mean_a=x.mean(), mean_b=y.mean(), mean_diff=d.mean(),
               ci_low=np.percentile(boot, 2.5), ci_high=np.percentile(boot, 97.5),
               median_diff=np.median(d),
               wins=int((d > tol).sum()), ties=int((np.abs(d) <= tol).sum()),
               losses=int((d < -tol).sum()))
    nz = d[np.abs(d) > tol]
    if len(nz) >= 1:
        try:
            w = stats.wilcoxon(x, y, zero_method="wilcox", alternative="two-sided")
            res["wilcoxon_W"], res["p"] = float(w.statistic), float(w.pvalue)
        except ValueError:
            pass
        ranks = stats.rankdata(np.abs(nz))
        res["rank_biserial"] = (ranks[nz > 0].sum() - ranks[nz < 0].sum()) / ranks.sum()
    else:
        res["p"] = 1.0
    sd = d.std(ddof=1) if n > 1 else np.nan
    res["cohen_dz"] = d.mean() / sd if sd and sd > 0 else np.nan
    res["cliffs_delta"] = cliffs_delta(x, y)
    res["cliffs_mag"] = cliffs_magnitude(res["cliffs_delta"])
    return res


def bootstrap_mean_ci(v, n_boot=N_BOOT, seed=SEED):
    v = np.asarray(v, float); v = v[~np.isnan(v)]
    if len(v) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    b = v[rng.integers(0, len(v), (n_boot, len(v)))].mean(axis=1)
    return v.mean(), np.percentile(b, 2.5), np.percentile(b, 97.5)


# ---------------- linear mixed-effects ----------------
def add_design_cols(df):
    d = df.copy()
    d["DT"] = (d["Family"] == "DeepTabular").astype(float)
    d["TF"] = (d["Family"] == "Transformer").astype(float)
    d["SMOTE_i"] = d["SMOTE"].astype(int)
    d["FS_i"] = d["FS"].astype(int)
    d["DatasetModel"] = d["Dataset"].astype(str) + "::" + d["Model"].astype(str)
    d["_g"] = 1
    return d


def fit_lme(d, formula, vc_terms, reml=True):
    """Crossed random intercepts via variance components (single dummy group).
    Tries several optimisers and keeps the first converged fit."""
    vc = {t: f"0 + C({t})" for t in vc_terms}
    model = smf.mixedlm(formula, d, groups="_g", vc_formula=vc, re_formula="0")
    best, last_err = None, None
    for method in ("lbfgs", "bfgs", "powell", "nm"):
        try:
            r = model.fit(reml=reml, method=method)
            r._fit_method = method
            if best is None or r.converged:
                best = r
            if r.converged:
                break
        except Exception as e:
            last_err = e
    if best is None:
        raise last_err
    best._formula = formula
    best._vc_terms = vc_terms
    best._reml = reml
    return best


def lme_fixed_table(fit):
    fe = fit.fe_params.index
    ci = fit.conf_int().loc[fe]
    return pd.DataFrame({"term": fe, "estimate": fit.fe_params.values,
                         "se": fit.bse_fe.values,
                         "z": (fit.fe_params / fit.bse_fe).values,
                         "p": fit.pvalues.loc[fe].values,
                         "ci_low": ci[0].values, "ci_high": ci[1].values})


def lme_contrast(fit, weights):
    """Linear combination of fixed effects, e.g. {'TF': 1, 'DT': -1}."""
    names = list(weights)
    w = np.array([weights[n] for n in names], float)
    b = fit.fe_params.loc[names].values
    V = fit.cov_params().loc[names, names].values
    est = float(w @ b)
    se = float(np.sqrt(w @ V @ w))
    z = est / se if se > 0 else np.nan
    p = 2 * (1 - stats.norm.cdf(abs(z))) if se > 0 else np.nan
    q = stats.norm.ppf(0.975)
    return dict(estimate=est, se=se, z=z, p=p, ci_low=est - q * se,
                ci_high=est + q * se)


def lme_variance_table(fit):
    rows = [{"component": n, "variance": v}
            for n, v in zip(fit.model.exog_vc.names, fit.vcomp)]
    rows.append({"component": "Residual", "variance": fit.scale})
    t = pd.DataFrame(rows)
    t["share"] = t["variance"] / t["variance"].sum()
    return t


def lme_report_text(fit, title):
    vt = lme_variance_table(fit)
    lines = [title, "=" * len(title),
             f"Formula (fixed effects): {fit._formula}",
             "Random effects: crossed random intercepts for "
             + ", ".join(fit._vc_terms)
             + " (statsmodels MixedLM, variance-component formulation, single dummy group)",
             f"Estimation: {'REML' if fit._reml else 'ML'}, optimiser = {fit._fit_method}",
             f"Converged: {fit.converged}",
             f"Observations: {int(fit.nobs)}",
             "", "Variance components:"]
    for _, r in vt.iterrows():
        lines.append(f"  {r['component']:<14s} variance = {r['variance']:.8f}  "
                     f"({100 * r['share']:.1f}% of total)")
    lines += ["", "Fixed effects:", lme_fixed_table(fit).to_string(index=False,
              float_format=lambda v: f"{v:.6f}"), "", str(fit.summary())]
    return "\n".join(lines)


def pareto_front(df, maximize=(), minimize=()):
    """Boolean Series: True if no other row is >= on all objectives and > on one."""
    vals = df[list(maximize) + list(minimize)].to_numpy(float).copy()
    k = len(maximize)
    vals[:, k:] *= -1                          # turn minimisation into maximisation
    eff = np.ones(len(df), bool)
    for i in range(len(df)):
        others = np.delete(vals, i, axis=0)
        dominated = np.any(np.all(others >= vals[i], axis=1)
                           & np.any(others > vals[i], axis=1))
        eff[i] = not dominated
    return pd.Series(eff, index=df.index)


print("Helpers ready.")

## 0. Data checks
Loads the four ablation folders and reports the analysis population.

In [ ]:
# ==========================================================================
# SECTION 0a -- Load the four ablation folders into one master table
#               + analysis-population report
# ==========================================================================

def load_mode_foldwise(mode):
    folder = STORE.mode_dir(mode)
    fpath = os.path.join(folder, FOLD_FILE_NAME)
    if os.path.exists(fpath):
        return pd.read_csv(fpath), "ALL_DATASETS file"
    # Fallback: rebuild from per-dataset <DS>_foldwise_metrics.csv (folder or zip)
    parts = []
    names = set()
    if os.path.isdir(folder):
        for item in os.listdir(folder):
            if item.endswith(".zip"):
                names.add(item[:-4])
            elif os.path.isdir(os.path.join(folder, item)):
                names.add(item)
    for ds in sorted(names):
        d = STORE.read_csv(mode, ds, f"{ds}_foldwise_metrics.csv")
        if d is not None:
            parts.append(d)
    if not parts:
        raise FileNotFoundError(
            f"No results found for Mode {mode} in {folder}. "
            "Check RESULTS_ROOT in the configuration cell.")
    return pd.concat(parts, ignore_index=True), "rebuilt from per-dataset files"


frames = []
for mode, (folder, smote, fs) in MODE_FOLDERS.items():
    df_m, src = load_mode_foldwise(mode)
    df_m["Mode"] = mode
    df_m["SMOTE"] = smote
    df_m["FS"] = fs
    frames.append(df_m)
    print(f"Mode {mode} ({folder}): {len(df_m)} rows  [{src}]")

master = pd.concat(frames, ignore_index=True)
n_before = len(master)
master = master.drop_duplicates(subset=["Mode", "Dataset", "Model", "Fold"], keep="last")
if len(master) < n_before:
    print(f"  removed {n_before - len(master)} duplicated (Mode, Dataset, Model, Fold) rows "
          "(caused by resumed runs)")
master["Family"] = master["Model"].map(MODEL_FAMILY).fillna(master["Family"])
master["IsEnsemble"] = master["Model"].isin(ENSEMBLES)
master["Repository"] = master["Dataset"].map(DATASET_REPO).fillna("Other")
master["ModelShort"] = master["Model"].map(disp)
DATASETS = sorted(master["Dataset"].unique(),
                  key=lambda d: (DATASET_REPO.get(d, "Z"), d))
master.to_csv(outpath("checks", "MasterFold_all_modes.csv"), index=False)

base = master[master["Model"].isin(BASE_LEARNERS)].copy()
ens = master[master["IsEnsemble"]].copy()

# ---------------- analysis population ----------------
pop_rows = []
for mode in MODE_FOLDERS:
    m = master[master["Mode"] == mode]
    b = m[m["Model"].isin(BASE_LEARNERS)]
    e = m[m["IsEnsemble"]]
    nd = m["Dataset"].nunique()
    pop_rows.append({
        "Mode": MODE_LABELS[mode], "Datasets": nd,
        "Base learners": b["Model"].nunique(), "Ensembles": e["Model"].nunique(),
        "Base rows": len(b), "Expected base rows": nd * len(BASE_LEARNERS) * N_FOLDS,
        "Ensemble rows": len(e), "Expected ensemble rows": nd * len(ENSEMBLES) * N_FOLDS,
        "Base rows with ROC-AUC": int(b["ROC_AUC"].notna().sum()),
    })
population = pd.DataFrame(pop_rows)
tot = population[["Base rows", "Expected base rows", "Ensemble rows",
                  "Expected ensemble rows", "Base rows with ROC-AUC"]].sum()
population.loc[len(population)] = {"Mode": "TOTAL (4 modes)", "Datasets": len(DATASETS),
                                   "Base learners": base["Model"].nunique(),
                                   "Ensembles": ens["Model"].nunique(), **tot.to_dict()}
save_table(population, "checks", "Table_analysis_population",
           caption="Number of fold-level observations entering each analysis.",
           label="tab:analysis_population")
print(population.to_string(index=False))

missing = []
for mode in MODE_FOLDERS:
    have = set(map(tuple, master.loc[master["Mode"] == mode,
                                     ["Dataset", "Model", "Fold"]].values))
    for ds in DATASETS:
        for mdl in ALL_MODELS:
            for f in range(1, N_FOLDS + 1):
                if (ds, mdl, f) not in have:
                    missing.append({"Mode": mode, "Dataset": ds, "Model": mdl, "Fold": f})
missing = pd.DataFrame(missing, columns=["Mode", "Dataset", "Model", "Fold"])
missing.to_csv(outpath("checks", "missing_model_fold_records.csv"), index=False)
if len(missing):
    print(f"\n!! {len(missing)} (mode, dataset, model, fold) records are missing -- "
          "see 00_data_checks/missing_model_fold_records.csv")
    print(missing.groupby(["Mode", "Model"]).size().rename("missing folds").to_string())

Dataset characteristics (imbalance ratio, instances, features, reduction %) are extracted from the result files.

In [ ]:
# ==========================================================================
# SECTION 0b -- Dataset characteristics extracted automatically
#   * instances / defective / IR      <- union of the 5 test folds' y_true
#                                        (= the de-duplicated dataset the models saw)
#   * number of features              <- feature_selection/fold_k/feature_importance.csv
#   * selected features & reduction % <- consensus selection, mean +/- sd over folds
# ==========================================================================

def dataset_label_counts(ds):
    """Read y_true from the 5 test folds of any base model (any mode)."""
    for mode in (1, 3, 2, 4):
        for mdl in BASE_LEARNERS:
            ys = []
            for f in range(1, N_FOLDS + 1):
                p = STORE.read_csv(mode, ds, f"{MODEL_DIRS[mdl]}/fold{f}_predictions.csv")
                if p is None:
                    break
                ys.append(p["y_true"].to_numpy())
            if len(ys) == N_FOLDS:
                fold_sizes = [len(y) for y in ys]
                y = np.concatenate(ys)
                return dict(n=len(y), n_def=int((y == 1).sum()),
                            fold_test_sizes=fold_sizes)
    return None


def read_fs_fold(mode, ds, fold):
    return STORE.read_csv(mode, ds, f"feature_selection/fold_{fold}/feature_importance.csv")


FS_FOLDS = {}          # (mode, ds) -> list of per-fold DataFrames
for mode in (3, 4):
    for ds in DATASETS:
        lst = [read_fs_fold(mode, ds, f) for f in range(1, N_FOLDS + 1)]
        if all(x is not None for x in lst):
            FS_FOLDS[(mode, ds)] = lst

ds_rows = []
TEST_FOLD_SIZE = {}    # (ds, fold) -> n test instances
for ds in DATASETS:
    lc = dataset_label_counts(ds)
    row = {"Dataset": ds, "Repository": DATASET_REPO.get(ds, "Other")}
    if lc:
        row.update(Instances=lc["n"], Defective=lc["n_def"],
                   Defective_pct=100 * lc["n_def"] / lc["n"],
                   IR=(lc["n"] - lc["n_def"]) / max(lc["n_def"], 1))
        for f, s in enumerate(lc["fold_test_sizes"], start=1):
            TEST_FOLD_SIZE[(ds, f)] = s
    fs_list = FS_FOLDS.get((4, ds)) or FS_FOLDS.get((3, ds))
    if fs_list:
        n_feat = [len(x) for x in fs_list]
        n_sel = [int(x["Consensus_Selected"].astype(bool).sum()) for x in fs_list]
        red = [100 * (1 - s / t) for s, t in zip(n_sel, n_feat)]
        row.update(Features=int(np.median(n_feat)),
                   Selected_mean=np.mean(n_sel), Selected_sd=np.std(n_sel, ddof=1),
                   Selected_min=min(n_sel), Selected_max=max(n_sel),
                   Reduction_pct_mean=np.mean(red), Reduction_pct_sd=np.std(red, ddof=1))
    ds_rows.append(row)

DATASET_INFO = pd.DataFrame(ds_rows).set_index("Dataset")
DATASET_IR = DATASET_INFO["IR"].to_dict() if "IR" in DATASET_INFO else {}
DATASET_REDUCTION_PCT = (DATASET_INFO["Reduction_pct_mean"].to_dict()
                         if "Reduction_pct_mean" in DATASET_INFO else {})
save_table(DATASET_INFO.reset_index().round(3), "checks", "Table_dataset_characteristics",
           caption="Dataset characteristics extracted from the revised results "
                   "(after exact-duplicate removal). Selected features are the "
                   "fold-wise SHAP-RF consensus subsets (mean, sd, range over 5 folds).",
           label="tab:dataset_characteristics")
print(DATASET_INFO.round(2).to_string())

if (3, DATASETS[0]) in FS_FOLDS and (4, DATASETS[0]) in FS_FOLDS:
    same = []
    for ds in DATASETS:
        if (3, ds) in FS_FOLDS and (4, ds) in FS_FOLDS:
            for a, b in zip(FS_FOLDS[(3, ds)], FS_FOLDS[(4, ds)]):
                same.append(set(a.loc[a["Consensus_Selected"].astype(bool), "Feature"])
                            == set(b.loc[b["Consensus_Selected"].astype(bool), "Feature"]))
    print(f"\nMode 3 vs Mode 4 selected the same subset in {sum(same)}/{len(same)} folds "
          "(expected: all, because selection happens before SMOTE).")

Optional raw-data audit for duplicates and conflicting labels.

In [ ]:
# ==========================================================================
# SECTION 0c -- OPTIONAL raw-data audit: duplicates & conflicting labels
#   Needs the raw .arff/.csv files in RAW_DATA_DIR (same loading rules as the
#   pipeline). Skipped automatically when the folder is absent.
# ==========================================================================
from scipy.io import arff as _arff

TARGET_CANDIDATES = ["Defective", "defects", "Defects", "DEFECT", "problems",
                     "class", "Class", "label", "Label"]


def _map_target(v):
    if isinstance(v, bytes):
        v = v.decode("utf-8")
    if isinstance(v, str):
        s = v.strip().lower()
        if s in ("y", "yes", "true", "1", "defective", "true_y", "buggy"):
            return 1
        if s in ("n", "no", "false", "0", "not_defective", "false_y", "clean"):
            return 0
        try:
            return 1 if float(s) > 0 else 0
        except ValueError:
            return np.nan
    try:
        return 1 if float(v) > 0 else 0
    except (TypeError, ValueError):
        return np.nan


def _load_raw(path):
    if path.lower().endswith(".arff"):
        data, _ = _arff.loadarff(path)
        df = pd.DataFrame(data)
    else:
        df = pd.read_csv(path)
    for c in df.columns:
        if df[c].dtype == object:
            df[c] = df[c].apply(lambda x: x.decode("utf-8") if isinstance(x, bytes) else x)
    tcol = next((c for c in TARGET_CANDIDATES if c in df.columns), df.columns[-1])
    df["target"] = df[tcol].apply(_map_target)
    if tcol != "target":
        df = df.drop(columns=[tcol])
    df = df.dropna(subset=["target"])
    df["target"] = df["target"].astype(int)
    feats = [c for c in df.columns if c != "target"]
    df[feats] = df[feats].apply(pd.to_numeric, errors="coerce")
    df = df.dropna(axis=1, how="all")
    return df


raw_files = {}
if os.path.isdir(RAW_DATA_DIR):
    for p in glob.glob(os.path.join(RAW_DATA_DIR, "**", "*"), recursive=True):
        stem, ext = os.path.splitext(os.path.basename(p))
        if ext.lower() in (".arff", ".csv") and stem in DATASETS and stem not in raw_files:
            raw_files[stem] = p

if not raw_files:
    print(f"RAW_DATA_DIR '{RAW_DATA_DIR}' not found or contains no dataset files -> "
          "duplicate / conflicting-label audit skipped.\n"
          "To run it, download the datasets as in the pipeline (Kaggle cells) and set "
          "RAW_DATA_DIR.")
else:
    audit = []
    for ds in DATASETS:
        if ds not in raw_files:
            continue
        df = _load_raw(raw_files[ds])
        feats = [c for c in df.columns if c != "target"]
        n_raw = len(df)
        n_exact_dup = int(df.duplicated().sum())
        dedup = df.drop_duplicates().reset_index(drop=True)
        n_labels = dedup.groupby(feats, dropna=False)["target"].transform("nunique")
        conflict_rows = int((n_labels > 1).sum())
        conflict_groups = int(dedup.loc[n_labels > 1, feats].drop_duplicates().shape[0])
        audit.append({
            "Dataset": ds, "Source file": os.path.basename(raw_files[ds]),
            "Raw instances": n_raw, "Raw features": len(feats),
            "Missing cells": int(df[feats].isna().sum().sum()),
            "Exact duplicates removed": n_exact_dup,
            "Instances after dedup": len(dedup),
            "Conflicting-label groups": conflict_groups,
            "Rows in conflicting groups": conflict_rows,
            "IR raw": (df["target"] == 0).sum() / max((df["target"] == 1).sum(), 1),
            "IR after dedup": (dedup["target"] == 0).sum() / max((dedup["target"] == 1).sum(), 1),
        })
    audit = pd.DataFrame(audit)
    save_table(audit.round(3), "checks", "Table_dataset_traceability_audit",
               caption="Raw-data audit: exact duplicates (removed by the pipeline) and "
                       "conflicting labels (identical feature vectors with different "
                       "labels; retained by the pipeline).",
               label="tab:traceability")
    print(audit.round(2).to_string(index=False))
    if "Instances" in DATASET_INFO:
        chk = audit.set_index("Dataset")["Instances after dedup"].reindex(DATASET_INFO.index)
        diff = (chk - DATASET_INFO["Instances"]).abs()
        print("\nConsistency with result files (instances after dedup):",
              "OK" if (diff.fillna(0) == 0).all() else f"MISMATCH\n{diff[diff > 0]}")

## 1. Table 1, Figure 1, Figure 2 for all four modes

In [ ]:
# ==========================================================================
# SECTION 1 -- Table 1 (13 base learners), Figure 1 (heatmap), Figure 2
#              (family boxplot) for EVERY ablation mode
# ==========================================================================

def build_table1(df_mode):
    agg = df_mode.groupby("Model")[METRICS + ["train_time_sec", "infer_time_sec"]].agg(["mean", "std"])
    agg.columns = [f"{m}_{s}" for m, s in agg.columns]
    agg = agg.reset_index()
    per_ds = df_mode.groupby(["Dataset", "Model"])["ROC_AUC"].mean().reset_index()
    per_ds["rank"] = per_ds.groupby("Dataset")["ROC_AUC"].rank(ascending=False, method="average")
    avg_rank = per_ds.groupby("Model")["rank"].mean().rename("Avg_Rank").reset_index()
    t = agg.merge(avg_rank, on="Model").sort_values("Avg_Rank").reset_index(drop=True)
    t.insert(1, "Family", t["Model"].map(MODEL_FAMILY))
    return t


def table1_latex(t, mode):
    smote, fs = MODE_FOLDERS[mode][1], MODE_FOLDERS[mode][2]
    cond = (f"{'with' if smote else 'without'} Borderline-SMOTE, "
            f"{'with' if fs else 'without'} SHAP-RF feature selection")
    nd = base.loc[base["Mode"] == mode, "Dataset"].nunique()
    L = [r"\begin{table*}[!t]", r"\centering",
         rf"\caption{{Mean $\pm$ std performance (\%) of the 13 base learners, pooled across "
         rf"{nd} datasets and {N_FOLDS}-fold cross-validation ({MODE_LABELS[mode]}: {cond}). "
         r"Models are sorted by average rank of ROC-AUC computed within each dataset.}",
         rf"\label{{tab:table1_mode{mode}}}", r"\resizebox{\textwidth}{!}{%",
         r"\begin{tabular}{l" + "c" * (len(METRICS) + 1) + "}", r"\toprule",
         " & ".join([r"\textbf{Model}"] + [rf"\textbf{{{METRIC_DISPLAY[m]}}}" for m in METRICS]
                    + [r"\textbf{Avg. Rank}"]) + r" \\", r"\midrule"]
    for _, r in t.iterrows():
        cells = [disp(r["Model"])] + [f"{r[m + '_mean'] * 100:.1f} $\\pm$ {r[m + '_std'] * 100:.1f}"
                                      for m in METRICS] + [f"{r['Avg_Rank']:.2f}"]
        L.append(" & ".join(cells) + r" \\")
    L += [r"\bottomrule", r"\end{tabular}%", "}", r"\end{table*}"]
    return "\n".join(L)


def plot_heatmap(df_mode, mode):
    pivot = df_mode.groupby(["Dataset", "Model"])["ROC_AUC"].mean().unstack("Model")
    pivot = pivot[[m for m in BASE_LEARNERS if m in pivot.columns]]
    pivot = pivot.reindex([d for d in DATASETS if d in pivot.index]).rename(columns=disp)
    vmin = max(0.4, np.floor(np.nanmin(pivot.values) * 20) / 20)
    fig, ax = plt.subplots(figsize=(14, 0.5 * len(pivot) + 2))
    ax.imshow(pivot.values, aspect="auto", cmap="Blues", vmin=vmin, vmax=1.0)
    ax.set_xticks(range(pivot.shape[1]))
    ax.set_xticklabels(pivot.columns, rotation=30, ha="right", fontsize=11, fontweight="bold")
    ax.set_yticks(range(pivot.shape[0]))
    ax.set_yticklabels(pivot.index, fontsize=11, fontweight="bold")
    thr = vmin + 0.6 * (1.0 - vmin)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            v = pivot.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f"{v * 100:.1f}", ha="center", va="center", fontsize=10,
                        fontweight="bold", color="white" if v > thr else "black")
    ax.set_xticks(np.arange(-0.5, pivot.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, pivot.shape[0], 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=0.8)
    ax.tick_params(which="minor", bottom=False, left=False)
    save_fig(fig, "table1", f"Figure1_heatmap_ROCAUC_mode{mode}", dpi=600)


def plot_family_box(df_mode, mode):
    data, labels = [], []
    for fam in FAMILY_ORDER:
        v = df_mode.loc[df_mode["Family"] == fam, "ROC_AUC"].dropna().values
        data.append(v)
        labels.append(f"{FAMILY_LABELS[fam]}\n(n={len(v)})")
    fig, ax = plt.subplots(figsize=(6, 4))
    bp = ax.boxplot(data, patch_artist=True, showmeans=True,
                    meanprops={"marker": "D", "markerfacecolor": "white",
                               "markeredgecolor": "black", "markersize": 6})
    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels)
    for patch, fam in zip(bp["boxes"], FAMILY_ORDER):
        patch.set_facecolor(FAMILY_COLORS[fam]); patch.set_alpha(0.6)
    ax.set_ylabel("Fold-level ROC-AUC")
    ax.set_title(MODE_LABELS[mode], fontsize=10)
    ax.yaxis.grid(True, linestyle="--", alpha=0.5); ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    save_fig(fig, "table1", f"Figure2_family_boxplot_mode{mode}")


TABLE1 = {}
for mode in MODE_FOLDERS:
    dm = base[base["Mode"] == mode]
    t = build_table1(dm)
    TABLE1[mode] = t
    t.to_csv(outpath("table1", f"Table1_base_learners_mode{mode}_raw.csv"), index=False)
    disp_t = pd.DataFrame({"Model": t["Model"].map(disp), "Family": t["Family"]})
    for m in METRICS:
        disp_t[METRIC_DISPLAY[m]] = [f"{a * 100:.1f} ± {b * 100:.1f}"
                                     for a, b in zip(t[m + "_mean"], t[m + "_std"])]
    disp_t["Avg. Rank"] = t["Avg_Rank"].round(2)
    disp_t.to_csv(outpath("table1", f"Table1_base_learners_mode{mode}_display.csv"), index=False)
    with open(outpath("table1", f"Table1_base_learners_mode{mode}.tex"), "w") as f:
        f.write(table1_latex(t, mode))
    plot_heatmap(dm, mode)
    plot_family_box(dm, mode)
    print(f"\n{MODE_LABELS[mode]}  ({len(dm)} fold-level rows)")
    print(disp_t.to_string(index=False))

# One compact table: mean ROC-AUC per model x mode
roc_by_mode = (base.groupby(["Model", "Mode"])["ROC_AUC"].mean().unstack("Mode") * 100)
roc_by_mode = roc_by_mode.reindex(BASE_LEARNERS).rename(columns=MODE_SHORT)
roc_by_mode.insert(0, "Family", [MODEL_FAMILY[m] for m in roc_by_mode.index])
roc_by_mode.index = roc_by_mode.index.map(disp)
save_table(roc_by_mode.round(2).reset_index(), "table1", "Table_ROCAUC_model_by_mode",
           caption="Mean ROC-AUC (\\%) of each base learner in the four ablation modes.",
           label="tab:roc_by_mode")

## 2. Feature-selection analysis

In [ ]:
# ==========================================================================
# SECTION 2 -- Feature-selection analysis
#   Uses feature_selection/fold_k/feature_importance.csv saved by the
#   pipeline (RF importance, mean|SHAP|, RF_Selected, SHAP_Selected,
#   Consensus_Selected).
#   (a) RF vs SHAP agreement inside each fold (same fitted forest)
#   (b) Stability of the selected subsets across the 5 training folds:
#       RF-only vs SHAP-only vs consensus
#   (c) Threshold sensitivity (percentile grid) -> subset size & stability
#   (d) Selection frequency of each feature
#   (e) Selection overhead (only if the pipeline recorded it)
# ==========================================================================

def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if (a | b) else 1.0


def kuncheva_lustgarten(a, b, d):
    """Kuncheva consistency index, Lustgarten et al. (2009) form for unequal sizes."""
    a, b = set(a), set(b)
    k1, k2, r = len(a), len(b), len(a & b)
    denom = min(k1, k2) - max(0, k1 + k2 - d)
    return (r - k1 * k2 / d) / denom if denom > 0 else np.nan


def nogueira_stability(subsets, features):
    """Nogueira, Sechidis & Brown (2018, JMLR) stability estimator."""
    M, d = len(subsets), len(features)
    Z = np.array([[f in s for f in features] for s in subsets], float)
    p = Z.mean(axis=0)
    s2 = M / (M - 1) * p * (1 - p)
    kbar = Z.sum(axis=1).mean()
    denom = (kbar / d) * (1 - kbar / d)
    return 1 - s2.mean() / denom if denom > 0 else np.nan


def stability_metrics(subsets, features):
    pairs = list(itertools.combinations(range(len(subsets)), 2))
    return dict(
        nogueira=nogueira_stability(subsets, features),
        jaccard=np.mean([jaccard(subsets[i], subsets[j]) for i, j in pairs]),
        kuncheva=np.nanmean([kuncheva_lustgarten(subsets[i], subsets[j], len(features))
                             for i, j in pairs]),
        size_mean=np.mean([len(s) for s in subsets]),
        size_sd=np.std([len(s) for s in subsets], ddof=1))


def select_at(fold_df, pct):
    """Reproduces select_features_fold(): >= percentile on RF and on SHAP, intersection,
    fallback = best combined rank if the intersection is empty."""
    rf = fold_df.set_index("Feature")["RF_Importance"]
    sh = fold_df.set_index("Feature")["SHAP_Importance"]
    rf_sel = set(rf[rf >= np.percentile(rf.values, pct)].index)
    sh_sel = set(sh[sh >= np.percentile(sh.values, pct)].index)
    cons = rf_sel & sh_sel
    if not cons:
        cons = {(rf.rank(pct=True) + sh.rank(pct=True)).idxmax()}
    return rf_sel, sh_sel, cons


FS_MODE = 4 if any(k[0] == 4 for k in FS_FOLDS) else 3
fs_ds = [ds for ds in DATASETS if (FS_MODE, ds) in FS_FOLDS]

if not fs_ds:
    print("No feature_selection/fold_k/feature_importance.csv files found -> Section 2 skipped.")
else:
    agree_rows, stab_rows, sens_rows, freq_rows, repro = [], [], [], [], []
    for ds in fs_ds:
        folds = FS_FOLDS[(FS_MODE, ds)]
        features = sorted(set().union(*[set(f["Feature"]) for f in folds]))
        # (a) agreement inside each fold
        for k, f in enumerate(folds, start=1):
            rf_s = set(f.loc[f["RF_Selected"].astype(bool), "Feature"])
            sh_s = set(f.loc[f["SHAP_Selected"].astype(bool), "Feature"])
            co_s = set(f.loc[f["Consensus_Selected"].astype(bool), "Feature"])
            rho = stats.spearmanr(f["RF_Importance"], f["SHAP_Importance"]).statistic
            agree_rows.append(dict(Dataset=ds, Fold=k, n_features=len(f),
                                   n_RF=len(rf_s), n_SHAP=len(sh_s), n_consensus=len(co_s),
                                   spearman_rho=rho, jaccard_RF_SHAP=jaccard(rf_s, sh_s),
                                   consensus_share_of_RF=len(co_s) / max(len(rf_s), 1)))
            # sanity check: recomputing the 50th-percentile rule reproduces the saved subset
            _, _, c50 = select_at(f, FS_PERCENTILE_USED)
            repro.append(c50 == co_s)
        # (b) stability across folds, per strategy (saved subsets)
        for strat, col in [("RF-only", "RF_Selected"), ("SHAP-only", "SHAP_Selected"),
                           ("Consensus (RF ∩ SHAP)", "Consensus_Selected")]:
            subs = [set(f.loc[f[col].astype(bool), "Feature"]) for f in folds]
            stab_rows.append(dict(Dataset=ds, Strategy=strat, **stability_metrics(subs, features)))
        # (c) threshold sensitivity
        for pct in THRESHOLD_GRID:
            sel = [select_at(f, pct) for f in folds]
            for si, strat in enumerate(["RF-only", "SHAP-only", "Consensus (RF ∩ SHAP)"]):
                subs = [s[si] for s in sel]
                m = stability_metrics(subs, features)
                sens_rows.append(dict(Dataset=ds, Percentile=pct, Strategy=strat,
                                      reduction_pct=100 * (1 - m["size_mean"] / len(features)),
                                      **m))
        # (d) selection frequency of each feature (consensus)
        subs = [set(f.loc[f["Consensus_Selected"].astype(bool), "Feature"]) for f in folds]
        for feat in features:
            freq_rows.append(dict(Dataset=ds, Feature=feat,
                                  folds_selected=sum(feat in s for s in subs)))

    print(f"Feature-selection files from Mode {FS_MODE}; 50th-percentile rule reproduces the "
          f"saved consensus subset in {sum(repro)}/{len(repro)} folds.")

    agree = pd.DataFrame(agree_rows)
    agree_ds = agree.groupby("Dataset").agg(
        n_features=("n_features", "median"), RF_selected=("n_RF", "mean"),
        SHAP_selected=("n_SHAP", "mean"), Consensus_selected=("n_consensus", "mean"),
        Spearman_RF_vs_SHAP=("spearman_rho", "mean"),
        Jaccard_RF_vs_SHAP=("jaccard_RF_SHAP", "mean"),
        Consensus_share_of_RF=("consensus_share_of_RF", "mean")).reindex(fs_ds)
    save_table(agree_ds.round(3).reset_index(), "fs", "Table_FS_RF_vs_SHAP_agreement",
               caption="Agreement between impurity importance and mean |SHAP| computed from "
                       "the same Random Forest inside each training fold (mean over 5 folds).",
               label="tab:fs_agreement")
    print("\n(a) RF vs SHAP agreement per dataset:\n", agree_ds.round(3).to_string())

    stab = pd.DataFrame(stab_rows)
    stab_wide = stab.pivot(index="Dataset", columns="Strategy",
                           values=["nogueira", "jaccard", "kuncheva", "size_mean"]).reindex(fs_ds)
    stab_wide.columns = [f"{m}__{s}" for m, s in stab_wide.columns]
    save_table(stab_wide.round(3).reset_index(), "fs", "Table_FS_stability_by_dataset",
               caption="Stability of the selected feature subsets across the 5 training folds "
                       "(Nogueira et al. 2018 estimator, mean pairwise Jaccard, "
                       "Kuncheva-Lustgarten index).", label="tab:fs_stability")
    stab_sum = stab.groupby("Strategy")[["nogueira", "jaccard", "kuncheva", "size_mean"]].agg(
        ["mean", "std", "min", "max"])
    stab_sum.columns = [f"{a}_{b}" for a, b in stab_sum.columns]
    save_table(stab_sum.round(3).reset_index(), "fs", "Table_FS_stability_summary",
               caption="Feature-selection stability summarised over datasets.",
               label="tab:fs_stability_summary")
    print("\n(b) Stability summary:\n", stab_sum.round(3).to_string())

    sens = pd.DataFrame(sens_rows)
    sens_sum = sens.groupby(["Strategy", "Percentile"])[
        ["size_mean", "reduction_pct", "nogueira", "jaccard"]].mean().round(3).reset_index()
    save_table(sens_sum, "fs", "Table_FS_threshold_sensitivity",
               caption="Sensitivity of subset size and cross-fold stability to the percentile "
                       "threshold (mean over datasets). The pipeline uses the 50th percentile.",
               label="tab:fs_threshold")
    print("\n(c) Threshold sensitivity (mean over datasets):\n", sens_sum.to_string(index=False))

    freq = pd.DataFrame(freq_rows)
    freq.to_csv(outpath("fs", "Table_FS_feature_selection_frequency.csv"), index=False)
    freq_hist = (freq.groupby(["Dataset", "folds_selected"]).size().unstack(fill_value=0)
                 .reindex(fs_ds))
    freq_hist.to_csv(outpath("fs", "Table_FS_frequency_histogram.csv"))

    # ---------- figures ----------
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    strats = ["RF-only", "SHAP-only", "Consensus (RF ∩ SHAP)"]
    x = np.arange(len(fs_ds)); w = 0.27
    for i, s in enumerate(strats):
        v = stab[stab["Strategy"] == s].set_index("Dataset").reindex(fs_ds)["nogueira"]
        axes[0].bar(x + (i - 1) * w, v, width=w, label=s)
    axes[0].set_xticks(x); axes[0].set_xticklabels(fs_ds, rotation=45, ha="right")
    axes[0].set_ylabel("Nogueira stability (1 = identical subsets)")
    axes[0].axhline(0.75, color="gray", ls=":", lw=0.8)
    axes[0].set_ylim(0, 1.0)
    axes[0].legend(fontsize=8, ncol=3, loc="lower center", bbox_to_anchor=(0.5, 1.0), frameon=False)
    for s in strats:
        g = sens_sum[sens_sum["Strategy"] == s]
        axes[1].plot(g["Percentile"], g["nogueira"], marker="o", label=s)
    axes[1].axvline(FS_PERCENTILE_USED, color="red", ls="--", lw=0.8)
    axes[1].set_xlabel("Percentile threshold"); axes[1].set_ylabel("Mean Nogueira stability")
    ax2 = axes[1].twinx()
    g = sens_sum[sens_sum["Strategy"] == "Consensus (RF ∩ SHAP)"]
    ax2.plot(g["Percentile"], g["reduction_pct"], color="gray", ls="--", marker="s",
             label="Consensus reduction %")
    ax2.set_ylabel("Feature reduction (%)", color="gray")
    h1, l1 = axes[1].get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    axes[1].legend(h1 + h2, l1 + l2, fontsize=8, ncol=2, loc="lower center",
                   bbox_to_anchor=(0.5, 1.0), frameon=False)
    for a in axes:
        a.spines["top"].set_visible(False)
    fig.tight_layout()
    save_fig(fig, "fs", "Figure_FS_stability_and_threshold")

    fig, ax = plt.subplots(figsize=(8, 0.35 * len(fs_ds) + 1.5))
    fh = freq_hist.div(freq_hist.sum(axis=1), axis=0) * 100
    left = np.zeros(len(fh))
    cmap = plt.get_cmap("Blues", N_FOLDS + 2)
    for c in sorted(fh.columns):
        ax.barh(fh.index, fh[c], left=left, color=cmap(c + 1), label=f"{c}/{N_FOLDS} folds")
        left += fh[c].values
    ax.set_xlabel("Share of features (%)")
    ax.legend(ncol=N_FOLDS + 1, fontsize=7, bbox_to_anchor=(0.5, 1.02), loc="lower center")
    fig.tight_layout()
    save_fig(fig, "fs", "Figure_FS_selection_frequency")

# (e) selection overhead -- only available if the pipeline stored it
fs_time_cols = [c for c in master.columns if c.lower() in ("fs_time_sec", "feature_selection_time_sec")]
if fs_time_cols:
    c = fs_time_cols[0]
    ov = master[master["FS"]].groupby(["Mode", "Dataset", "Fold"])[c].first()
    ovt = ov.groupby("Dataset").agg(["mean", "std"]).reindex(DATASETS)
    save_table(ovt.round(3).reset_index(), "fs", "Table_FS_overhead",
               caption="Feature-selection overhead per training fold (seconds).",
               label="tab:fs_overhead")
else:
    print("\n(e) No feature-selection timing column in the results -> overhead table skipped.")

## 3. Effects of feature selection and Borderline-SMOTE

In [ ]:
# ==========================================================================
# SECTION 3 -- Effect of feature selection (FS) and of Borderline-SMOTE
#   Overall / by family / by model / per dataset / 4-mode ablation, with
#   reduction % and IR extracted from the results, plus paired per-dataset tests
#   (Wilcoxon, Holm, bootstrap CI, Cliff's delta) and the FS x SMOTE
#   interaction contrast with CI.
# ==========================================================================
EFFECT_METRICS = ["ROC_AUC", "F1", "MCC", "Accuracy", "Precision", "Recall",
                  "train_time_sec", "infer_time_sec"]


def factor_effect(factor, label_on, label_off, tag):
    on_off = {False: label_off, True: label_on}
    # overall (pooled over the other factor, datasets, folds, base learners)
    t = base.groupby(factor)[EFFECT_METRICS].mean().rename(index=on_off).T
    t[f"Delta ({label_on} - {label_off})"] = t[label_on] - t[label_off]
    save_table(t.round(4).reset_index(names="Metric"), "effects", f"Table_{tag}_effect_overall",
               caption=f"Overall effect of {label_on} (fold-level, 13 base learners, pooled "
                       "over the other preprocessing factor).", label=f"tab:{tag.lower()}_overall")
    # conditional on the other factor
    other = "SMOTE" if factor == "FS" else "FS"
    cond = base.groupby([other, factor])[["ROC_AUC", "F1", "MCC"]].mean().unstack(factor)
    cond_t = pd.DataFrame({f"{m}_Delta": cond[(m, True)] - cond[(m, False)]
                           for m in ["ROC_AUC", "F1", "MCC"]})
    cond_t.index = [f"{other}={'on' if v else 'off'}" for v in cond_t.index]
    save_table(cond_t.round(4).reset_index(names="Condition"), "effects",
               f"Table_{tag}_effect_conditional",
               caption=f"Effect of {label_on} within each level of {other}.",
               label=f"tab:{tag.lower()}_conditional")
    # by family
    fam = base.groupby(["Family", factor])[["ROC_AUC", "F1", "MCC", "train_time_sec"]].mean().unstack(factor)
    fam_t = pd.DataFrame(index=fam.index)
    for m in ["ROC_AUC", "F1", "MCC", "train_time_sec"]:
        fam_t[f"{m}_{label_off}"] = fam[(m, False)]
        fam_t[f"{m}_{label_on}"] = fam[(m, True)]
        fam_t[f"{m}_Delta"] = fam[(m, True)] - fam[(m, False)]
    save_table(fam_t.round(4).reset_index(), "effects", f"Table_{tag}_effect_by_family",
               caption=f"Effect of {label_on} by model family.", label=f"tab:{tag.lower()}_family")
    # by model
    mod = base.groupby(["Model", factor])[["ROC_AUC", "F1", "MCC", "train_time_sec"]].mean().unstack(factor)
    mod_t = pd.DataFrame({f"{m}_Delta": mod[(m, True)] - mod[(m, False)]
                          for m in ["ROC_AUC", "F1", "MCC", "train_time_sec"]})
    mod_t["Family"] = mod_t.index.map(MODEL_FAMILY)
    mod_t.index = mod_t.index.map(disp)
    mod_t = mod_t.sort_values("ROC_AUC_Delta", ascending=False)
    save_table(mod_t.round(4).reset_index(names="Model"), "effects", f"Table_{tag}_effect_by_model",
               caption=f"Effect of {label_on} by base learner.", label=f"tab:{tag.lower()}_model")
    # per dataset
    pdm = base.groupby(["Dataset", factor])[["ROC_AUC", "F1", "MCC"]].mean().unstack(factor) * 100
    dd = pd.DataFrame(index=pdm.index)
    for m in ["ROC_AUC", "F1", "MCC"]:
        dd[f"{m}_{label_off}"] = pdm[(m, False)]
        dd[f"{m}_{label_on}"] = pdm[(m, True)]
        dd[f"{m}_Delta"] = pdm[(m, True)] - pdm[(m, False)]
    dd["Reduction_pct"] = dd.index.map(DATASET_REDUCTION_PCT)
    dd["IR"] = dd.index.map(DATASET_IR)
    dd["Repository"] = dd.index.map(DATASET_REPO)
    dd = dd.sort_values("ROC_AUC_Delta", ascending=False)
    save_table(dd.round(3).reset_index(), "effects", f"Table_{tag}_effect_by_dataset",
               caption=f"Per-dataset effect of {label_on} (percentage points), averaged over "
                       "base learners, folds and the other factor.",
               label=f"tab:{tag.lower()}_dataset")
    # paired per-dataset tests: overall and per family
    tests = []
    for fam_name in ["All base learners"] + FAMILY_ORDER:
        sub = base if fam_name == "All base learners" else base[base["Family"] == fam_name]
        for m in ["ROC_AUC", "F1", "MCC"]:
            p = sub.groupby(["Dataset", factor])[m].mean().unstack(factor)
            r = paired_compare(p[True].values, p[False].values)
            tests.append(dict(Group=fam_name, Metric=m, **r))
    tests = pd.DataFrame(tests)
    tests["p_holm"] = holm_adjust(tests["p"].values)
    save_table(tests.round(4), "effects", f"Table_{tag}_effect_paired_tests",
               caption=f"Paired per-dataset comparison {label_on} vs {label_off} (n = datasets). "
                       "Wilcoxon signed-rank, Holm-adjusted over all rows; 95\\% bootstrap CI "
                       "of the mean difference; Cliff's delta.",
               label=f"tab:{tag.lower()}_tests")
    print(f"\n===== {label_on} vs {label_off} =====")
    print(t.round(4).to_string())
    print("\nConditional:\n", cond_t.round(4).to_string())
    print("\nPaired per-dataset tests:\n",
          tests[["Group", "Metric", "n", "mean_diff", "ci_low", "ci_high", "p", "p_holm",
                 "cliffs_delta", "cliffs_mag"]].round(4).to_string(index=False))
    return t, fam_t, mod_t, dd, tests


fs_overall, fs_fam, fs_mod, fs_ds_delta, fs_tests = factor_effect("FS", "FS", "No-FS", "FS")
sm_overall, sm_fam, sm_mod, sm_ds_delta, sm_tests = factor_effect("SMOTE", "SMOTE", "No-SMOTE", "SMOTE")

# correlations with dataset properties
corr_rows = []
for x_name, x_map, dd, lab in [("Reduction_pct", DATASET_REDUCTION_PCT, fs_ds_delta, "FS"),
                               ("IR", DATASET_IR, sm_ds_delta, "SMOTE"),
                               ("log(IR)", {k: np.log(v) for k, v in DATASET_IR.items()}, sm_ds_delta, "SMOTE")]:
    xv = np.asarray([x_map.get(i, np.nan) for i in dd.index], float)
    ok = ~np.isnan(xv)
    if ok.sum() >= 4:
        pr = stats.pearsonr(xv[ok], dd["ROC_AUC_Delta"].values[ok])
        sr = stats.spearmanr(xv[ok], dd["ROC_AUC_Delta"].values[ok])
        corr_rows.append(dict(Effect=lab, Variable=x_name, n=int(ok.sum()),
                              Pearson_r=pr.statistic, Pearson_p=pr.pvalue,
                              Spearman_rho=sr.statistic, Spearman_p=sr.pvalue))
corr_tab = pd.DataFrame(corr_rows)
save_table(corr_tab.round(4), "effects", "Table_effect_vs_dataset_properties",
           caption="Correlation between per-dataset ROC-AUC change and dataset properties.",
           label="tab:effect_corr")
print("\n", corr_tab.round(4).to_string(index=False))

# ---------- four-mode ablation table ----------
table4 = base.groupby("Mode")[["ROC_AUC", "F1", "MCC", "Accuracy", "Recall", "Precision",
                               "train_time_sec", "infer_time_sec"]].mean()
ci_rows = []
for mode in MODE_FOLDERS:
    v = base[base["Mode"] == mode].groupby("Dataset")["ROC_AUC"].mean()
    mu, lo, hi = bootstrap_mean_ci(v.values)
    ci_rows.append((lo, hi))
table4["ROC_AUC_CI_low"] = [c[0] for c in ci_rows]
table4["ROC_AUC_CI_high"] = [c[1] for c in ci_rows]
table4.index = table4.index.map(MODE_LABELS)
save_table(table4.round(4).reset_index(), "effects", "Table4_mode_comparison",
           caption="Average performance and cost across the four ablation modes (13 base learners; "
                   "ROC-AUC 95\\% CI bootstrapped over datasets).", label="tab:mode_comparison")
print("\n", table4.round(4).to_string())

# ---------- FS x SMOTE interaction with uncertainty ----------
cell = base.groupby(["Dataset", "Mode"])["ROC_AUC"].mean().unstack("Mode")
inter = (cell[4] - cell[2]) - (cell[3] - cell[1])           # (FS effect | SMOTE) - (FS effect | no SMOTE)
r = paired_compare(((cell[4] - cell[2])).values, ((cell[3] - cell[1])).values)
syn = pd.DataFrame([dict(Contrast="(M4 - M2) - (M3 - M1)  [FS x SMOTE interaction]", **r)])
save_table(syn.round(5), "effects", "Table_FS_SMOTE_synergy_contrast",
           caption="Per-dataset FS x SMOTE interaction contrast on ROC-AUC with Wilcoxon test "
                   "and 95\\% bootstrap CI (n = datasets).", label="tab:synergy")

# ---------- figures ----------
def grouped_bar_by_model(factor, label_on, label_off, name):
    p = base.groupby(["Model", factor])["ROC_AUC"].mean().unstack() * 100
    p = p.sort_values(True, ascending=False)
    fig, ax = plt.subplots(figsize=(10, 3.2))
    x = np.arange(len(p)); w = 0.38
    ax.bar(x - w / 2, p[False], width=w, label=label_off, color="#003f5c")
    ax.bar(x + w / 2, p[True], width=w, label=label_on, color="#4289ed")
    ax.set_xticks(x); ax.set_xticklabels([disp(m) for m in p.index], rotation=25, ha="right")
    lo, hi = np.nanmin(p.values), np.nanmax(p.values)
    ax.set_ylim(np.floor(lo - 2), np.ceil(hi + 1))
    ax.set_ylabel("Mean ROC-AUC (%)")
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.legend()
    save_fig(fig, "effects", name)


grouped_bar_by_model("FS", "FS", "No-FS", "Figure_FS_grouped_bar_by_model")
grouped_bar_by_model("SMOTE", "SMOTE", "No-SMOTE", "Figure_SMOTE_grouped_bar_by_model")

vals = base.groupby("Mode")[["ROC_AUC", "F1", "MCC"]].mean().loc[[1, 2, 3, 4]] * 100
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(4); w = 0.25
for i, (m, c) in enumerate(zip(["ROC_AUC", "F1", "MCC"], ["#003f5c", "#ffa600", "#aaaaaa"])):
    ax.bar(x + (i - 1) * w, vals[m], width=w, label=METRIC_DISPLAY[m], color=c)
ax.set_xticks(x); ax.set_xticklabels([MODE_LABELS[k].replace(": ", "\n") for k in [1, 2, 3, 4]])
ax.set_ylabel("Score (%)")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
ax.legend()
save_fig(fig, "effects", "Figure_Mode_comparison_bar")


def scatter_delta(dd, xcol, xlabel, ylabel, name):
    try:
        from adjustText import adjust_text
    except ImportError:
        adjust_text = None
    fig, ax = plt.subplots(figsize=(7, 5.5))
    texts = []
    for repo, grp in dd.dropna(subset=[xcol]).groupby("Repository"):
        ax.scatter(grp[xcol], grp["ROC_AUC_Delta"], s=70, alpha=0.85, label=repo,
                   color={"NASA": "#003f5c", "AEEEM": "#ffa600"}.get(repo, "gray"))
        for n_, row in grp.iterrows():
            texts.append(ax.text(row[xcol], row["ROC_AUC_Delta"], n_, fontsize=8))
    if adjust_text and texts:
        adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color="gray", lw=0.5))
    ax.axhline(0, color="gray", lw=0.8, ls="--")
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.legend(title="Repository")
    save_fig(fig, "effects", name)


scatter_delta(fs_ds_delta, "Reduction_pct", "Feature reduction (%, mean over folds)",
              "ROC-AUC change, FS $-$ No-FS (pp)", "Figure_ReductionPct_vs_ROCAUC_delta")
scatter_delta(sm_ds_delta, "IR", "Imbalance ratio (IR)",
              "ROC-AUC change, SMOTE $-$ No-SMOTE (pp)", "Figure_IR_vs_ROCAUC_delta")
print("\nSection 3 tables and figures written to", os.path.join(OUT_ROOT, SECTIONS["effects"]))

## 4. Statistics
### 4a. Family LMEs per mode, Holm-adjusted contrasts

In [ ]:
# ==========================================================================
# SECTION 4a -- Family comparison per mode: linear mixed-effects models
#   * fold level   : ROC_AUC ~ Family + (1|Dataset) + (1|Model) + (1|Dataset:Model)
#                    -> the Dataset:Model term absorbs the dependence of the
#                       5 folds that share test partitions / training data
#   * dataset level: fold-averaged scores, (1|Dataset) + (1|Model)  (robustness)
#   * contrasts DT-C, TF-C, TF-DT; Holm over all modes x contrasts per level
#   * 95% CI, effect in pp, practical-equivalence check vs +/-PRACTICAL_DELTA
# ==========================================================================
CONTRASTS = {"DeepTabular - Classical": {"DT": 1.0},
             "Transformer - Classical": {"TF": 1.0},
             "Transformer - DeepTabular": {"TF": 1.0, "DT": -1.0}}
LME_FITS = {}
lme_rows, vc_rows = [], []

for level in ("fold", "dataset"):
    for mode in MODE_FOLDERS:
        d = base[(base["Mode"] == mode) & base[PRIMARY_METRIC].notna()]
        if level == "dataset":
            d = d.groupby(["Dataset", "Model", "Family", "SMOTE", "FS"], as_index=False)[PRIMARY_METRIC].mean()
            vcs = ["Dataset", "Model"]
        else:
            vcs = ["Dataset", "Model", "DatasetModel"]
        d = add_design_cols(d)
        fit = fit_lme(d, f"{PRIMARY_METRIC} ~ DT + TF", vcs)
        LME_FITS[(level, mode)] = fit
        with open(outpath("stats", f"LME_family_mode{mode}_{level}level.txt"), "w") as f:
            f.write(lme_report_text(fit, f"Family LME -- {MODE_LABELS[mode]} -- {level}-level data"))
        for cname, w in CONTRASTS.items():
            c = lme_contrast(fit, w)
            lme_rows.append(dict(Level=level, Mode=mode, Contrast=cname, n_obs=int(fit.nobs),
                                 converged=fit.converged, **c))
        for _, r in lme_variance_table(fit).iterrows():
            vc_rows.append(dict(Level=level, Mode=mode, **r.to_dict()))

lme_tab = pd.DataFrame(lme_rows)
lme_tab["p_holm"] = np.nan
for level in lme_tab["Level"].unique():
    m = lme_tab["Level"] == level
    lme_tab.loc[m, "p_holm"] = holm_adjust(lme_tab.loc[m, "p"].values)


def interpret(r):
    if r["p_holm"] < ALPHA:
        sig = "difference detected (Holm)"
    else:
        sig = "no difference detected"
    if r["ci_low"] > -PRACTICAL_DELTA_AUC and r["ci_high"] < PRACTICAL_DELTA_AUC:
        prac = f"CI within ±{PRACTICAL_DELTA_AUC:g} (practically equivalent)"
    elif r["ci_high"] <= -PRACTICAL_DELTA_AUC or r["ci_low"] >= PRACTICAL_DELTA_AUC:
        prac = f"CI beyond ±{PRACTICAL_DELTA_AUC:g} (practically relevant)"
    else:
        prac = "inconclusive w.r.t. practical relevance"
    return pd.Series({"Statistical": sig, "Practical": prac})


lme_tab[["Statistical", "Practical"]] = lme_tab.apply(interpret, axis=1)
lme_tab["Mode"] = lme_tab["Mode"].map(MODE_LABELS)
lme_tab["estimate_pp"] = lme_tab["estimate"] * 100
save_table(lme_tab.round(6), "stats", "Table_LME_family_contrasts",
           caption="Family contrasts from linear mixed-effects models of ROC-AUC per ablation mode. "
                   "Estimates are ROC-AUC differences; 95\\% Wald CIs; p-values Holm-adjusted over "
                   "4 modes x 3 contrasts within each data level.",
           label="tab:lme_contrasts")
vc_tab = pd.DataFrame(vc_rows)
vc_tab["Mode"] = vc_tab["Mode"].map(MODE_LABELS)
save_table(vc_tab, "stats", "Table_LME_variance_components", float_format="%.8f",
           caption="Variance components of the family LMEs.", label="tab:lme_vc")
print(lme_tab[["Level", "Mode", "Contrast", "estimate_pp", "ci_low", "ci_high", "p", "p_holm",
               "Statistical", "Practical", "converged"]].round(4).to_string(index=False))

### 4b. Residual diagnostics

In [ ]:
# ==========================================================================
# SECTION 4b -- Residual diagnostics of the fold-level family LMEs
# ==========================================================================
diag_rows = []
fig, axes = plt.subplots(4, 3, figsize=(13, 14))
for i, mode in enumerate(MODE_FOLDERS):
    fit = LME_FITS[("fold", mode)]
    res = np.asarray(fit.resid)
    fitted = np.asarray(fit.fittedvalues)
    fam = fit.model.data.frame["Family"].values
    ax = axes[i, 0]
    ax.scatter(fitted, res, s=6, alpha=0.4)
    ax.axhline(0, color="red", lw=0.8)
    ax.set_title(f"{MODE_SHORT[mode]}: residuals vs fitted", fontsize=9)
    ax.set_xlabel("Fitted ROC-AUC"); ax.set_ylabel("Residual")
    stats.probplot(res, dist="norm", plot=axes[i, 1])
    axes[i, 1].set_title(f"{MODE_SHORT[mode]}: normal Q-Q", fontsize=9)
    axes[i, 2].boxplot([res[fam == f] for f in FAMILY_ORDER])
    axes[i, 2].set_xticks(range(1, 4)); axes[i, 2].set_xticklabels(FAMILY_ORDER)
    axes[i, 2].set_title(f"{MODE_SHORT[mode]}: residuals by family", fontsize=9)
    sh = stats.shapiro(res[:5000]) if len(res) >= 3 else None
    lev = stats.levene(*[res[fam == f] for f in FAMILY_ORDER if (fam == f).sum() > 1])
    diag_rows.append(dict(Mode=MODE_LABELS[mode], n=len(res), resid_sd=res.std(ddof=1),
                          skewness=stats.skew(res), excess_kurtosis=stats.kurtosis(res),
                          shapiro_W=sh.statistic if sh else np.nan,
                          shapiro_p=sh.pvalue if sh else np.nan,
                          levene_family_W=lev.statistic, levene_family_p=lev.pvalue,
                          converged=fit.converged, optimiser=fit._fit_method))
fig.tight_layout()
save_fig(fig, "stats", "Figure_LME_residual_diagnostics")
diag = pd.DataFrame(diag_rows)
save_table(diag.round(4), "stats", "Table_LME_residual_diagnostics",
           caption="Residual diagnostics of the fold-level family LMEs (Shapiro-Wilk normality, "
                   "Levene homogeneity across families).", label="tab:lme_diag")
print(diag.round(4).to_string(index=False))

### 4c. Paired dataset-level tests, Friedman / Nemenyi

In [ ]:
# ==========================================================================
# SECTION 4c -- Paired dataset-level analyses (robustness check, effect sizes)
#   * family means per dataset -> Wilcoxon signed-rank (n = datasets),
#     Holm over 4 modes x 3 contrasts, bootstrap CI, Cliff's delta,
#     matched-pairs rank-biserial r
#   * Friedman test over the 13 base learners + Nemenyi post-hoc and
#     critical-difference (CD) diagram per mode (Demsar 2006)
# ==========================================================================
fam_rows = []
for mode in MODE_FOLDERS:
    p = (base[base["Mode"] == mode].groupby(["Dataset", "Family"])[PRIMARY_METRIC].mean()
         .unstack("Family"))
    for cname, (a, b) in {"DeepTabular - Classical": ("DeepTabular", "Classical"),
                          "Transformer - Classical": ("Transformer", "Classical"),
                          "Transformer - DeepTabular": ("Transformer", "DeepTabular")}.items():
        r = paired_compare(p[a].values, p[b].values)
        fam_rows.append(dict(Mode=MODE_LABELS[mode], Contrast=cname, **r))
fam_paired = pd.DataFrame(fam_rows)
fam_paired["p_holm"] = holm_adjust(fam_paired["p"].values)
fam_paired["mean_diff_pp"] = fam_paired["mean_diff"] * 100
save_table(fam_paired.round(5), "stats", "Table_family_paired_dataset_level",
           caption="Paired dataset-level family comparison of ROC-AUC (family mean per dataset; "
                   "n = datasets). Wilcoxon signed-rank with Holm adjustment over 12 tests; "
                   "95\\% bootstrap CI of the mean difference; Cliff's delta (Romano thresholds) "
                   "and matched-pairs rank-biserial correlation.",
           label="tab:family_paired")
print(fam_paired[["Mode", "Contrast", "n", "mean_diff_pp", "ci_low", "ci_high", "wins", "losses",
                  "p", "p_holm", "cliffs_delta", "cliffs_mag", "rank_biserial"]].round(4)
      .to_string(index=False))

# ---------------- Friedman + Nemenyi ----------------
try:
    import scikit_posthocs as sp
except ImportError:
    sp = None
fried_rows = []
for mode in MODE_FOLDERS:
    piv = (base[base["Mode"] == mode].groupby(["Dataset", "Model"])[PRIMARY_METRIC].mean()
           .unstack("Model").reindex(columns=BASE_LEARNERS).dropna())
    N, k = piv.shape
    if N < 3:
        continue
    fr = stats.friedmanchisquare(*[piv[c].values for c in piv.columns])
    ranks = piv.rank(axis=1, ascending=False).mean().sort_values()
    q = studentized_range.ppf(1 - ALPHA, k, np.inf) / np.sqrt(2)
    cd = q * np.sqrt(k * (k + 1) / (6 * N))
    iman = (N - 1) * fr.statistic / (N * (k - 1) - fr.statistic)   # Iman-Davenport F
    iman_p = 1 - stats.f.cdf(iman, k - 1, (k - 1) * (N - 1))
    fried_rows.append(dict(Mode=MODE_LABELS[mode], N_datasets=N, k_models=k,
                           chi2=fr.statistic, p=fr.pvalue, ImanDavenport_F=iman,
                           ImanDavenport_p=iman_p, CD=cd,
                           best=disp(ranks.index[0]), best_rank=ranks.iloc[0]))
    rk = ranks.rename(index=disp).reset_index()
    rk.columns = ["Model", "Average rank"]
    rk.to_csv(outpath("stats", f"Table_Friedman_avg_ranks_mode{mode}.csv"), index=False)
    if sp is not None:
        nem = sp.posthoc_nemenyi_friedman(piv.values)
        nem.index = nem.columns = [disp(c) for c in piv.columns]
        nem.round(4).to_csv(outpath("stats", f"Table_Nemenyi_pvalues_mode{mode}.csv"))
    # CD diagram (rank axis, CD bar, models within CD of the best are bold)
    fig, ax = plt.subplots(figsize=(9, 0.32 * k + 1.2))
    y = np.arange(k)[::-1]
    for yi, (m, rv) in zip(y, ranks.items()):
        ax.plot(rv, yi, "o", color=FAMILY_COLORS[MODEL_FAMILY[m]])
        ax.text(rv + 0.12, yi, disp(m), va="center", fontsize=9,
                fontweight="bold" if rv - ranks.iloc[0] <= cd else "normal")
    ax.axvspan(ranks.iloc[0], ranks.iloc[0] + cd, color="gray", alpha=0.15,
               label=f"CD = {cd:.2f} (Nemenyi, α={ALPHA})")
    ax.set_yticks([]); ax.set_xlabel("Average rank of ROC-AUC (lower is better)")
    ax.set_xlim(0.5, k + 1.5)
    ax.set_title(f"{MODE_LABELS[mode]} -- Friedman p={fmt_p(fr.pvalue)}", fontsize=10)
    ax.legend(loc="upper right", fontsize=8)
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    save_fig(fig, "stats", f"Figure_CD_diagram_mode{mode}")
fried = pd.DataFrame(fried_rows)
save_table(fried.round(5), "stats", "Table_Friedman_tests",
           caption="Friedman test over the 13 base learners per mode (blocks = datasets), "
                   "Iman-Davenport correction and Nemenyi critical difference.",
           label="tab:friedman")
print("\n", fried.round(4).to_string(index=False))

### 4d. Family × SMOTE × FS interaction model

In [ ]:
# ==========================================================================
# SECTION 4d -- Architecture x preprocessing interaction
#   One LME over all four modes:
#     ROC_AUC ~ (DT + TF) * SMOTE * FS
#              + (1|Dataset) + (1|Model) + (1|Dataset:Model)
#   * DT:SMOTE / TF:SMOTE -> does SMOTE change the family gap?
#   * DT:FS   / TF:FS     -> does FS change the family gap?
#   * SMOTE:FS            -> FS x SMOTE interaction (for Classical)
#   Family gaps in each mode are derived from the same model (contrasts).
# ==========================================================================
INTER_FITS = {}
inter_rows, gap_rows = [], []
for level in ("fold", "dataset"):
    d = base[base[PRIMARY_METRIC].notna()]
    if level == "dataset":
        d = d.groupby(["Dataset", "Model", "Family", "Mode", "SMOTE", "FS"], as_index=False)[PRIMARY_METRIC].mean()
    d = add_design_cols(d)
    fit = fit_lme(d, f"{PRIMARY_METRIC} ~ (DT + TF) * SMOTE_i * FS_i",
                  ["Dataset", "Model", "DatasetModel"])
    INTER_FITS[level] = fit
    with open(outpath("stats", f"LME_interaction_{level}level.txt"), "w") as f:
        f.write(lme_report_text(fit, f"Family x SMOTE x FS interaction LME -- {level}-level data"))
    ft = lme_fixed_table(fit)
    ft = ft[ft["term"] != "Intercept"].copy()
    ft["p_holm"] = holm_adjust(ft["p"].values)
    ft.insert(0, "Level", level)
    inter_rows.append(ft)
    for mode, (folder, s, f_) in MODE_FOLDERS.items():
        s, f_ = int(s), int(f_)
        for fam, key in (("DeepTabular", "DT"), ("Transformer", "TF")):
            w = {key: 1.0}
            if s: w[f"{key}:SMOTE_i"] = 1.0
            if f_: w[f"{key}:FS_i"] = 1.0
            if s and f_: w[f"{key}:SMOTE_i:FS_i"] = 1.0
            c = lme_contrast(fit, w)
            gap_rows.append(dict(Level=level, Mode=MODE_LABELS[mode], ModeNum=mode,
                                 Contrast=f"{fam} - Classical", **c))

inter_tab = pd.concat(inter_rows, ignore_index=True)
save_table(inter_tab.round(6), "stats", "Table_LME_interaction_terms",
           caption="Fixed effects of the Family x SMOTE x FS mixed model on ROC-AUC (reference: "
                   "Classical, no SMOTE, no FS). Holm adjustment over all non-intercept terms "
                   "within each data level.", label="tab:lme_interaction")
gap_tab = pd.DataFrame(gap_rows)
save_table(gap_tab.round(6), "stats", "Table_LME_family_gap_by_mode_from_interaction_model",
           caption="Family gap (vs Classical) in each mode derived from the interaction model.",
           label="tab:lme_gaps")
print(inter_tab[["Level", "term", "estimate", "ci_low", "ci_high", "p", "p_holm"]].round(5)
      .to_string(index=False))

# interaction plot: family mean ROC-AUC per mode, 95% bootstrap CI over datasets
fig, ax = plt.subplots(figsize=(7, 4.2))
for j, fam in enumerate(FAMILY_ORDER):
    mus, los, his = [], [], []
    for mode in MODE_FOLDERS:
        v = base[(base["Mode"] == mode) & (base["Family"] == fam)].groupby("Dataset")[PRIMARY_METRIC].mean()
        mu, lo, hi = bootstrap_mean_ci(v.values)
        mus.append(mu * 100); los.append(lo * 100); his.append(hi * 100)
    xs = np.arange(4) + (j - 1) * 0.08
    ax.errorbar(xs, mus, yerr=[np.array(mus) - los, np.array(his) - mus], marker="o",
                capsize=3, label=FAMILY_LABELS[fam], color=FAMILY_COLORS[fam])
ax.set_xticks(range(4)); ax.set_xticklabels([MODE_LABELS[m].replace(": ", "\n") for m in MODE_FOLDERS])
ax.set_ylabel("Mean ROC-AUC (%) [95% CI over datasets]")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
ax.legend()
save_fig(fig, "stats", "Figure_family_by_mode_interaction_plot")

### 4e. Forest plot computed from the fitted models

In [ ]:
# ==========================================================================
# SECTION 4e -- Forest plot of the family effects, computed from the
#               fitted fold-level mixed models
# ==========================================================================
fp = lme_tab[(lme_tab["Level"] == "fold") & (lme_tab["Contrast"] != "Transformer - DeepTabular")].copy()
fp = fp.iloc[::-1].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(7.5, 5))
for i, r in fp.iterrows():
    col = FAMILY_COLORS["DeepTabular" if r["Contrast"].startswith("Deep") else "Transformer"]
    ax.errorbar(r["estimate"], i, xerr=[[r["estimate"] - r["ci_low"]], [r["ci_high"] - r["estimate"]]],
                fmt="o", capsize=4, color=col)
    ax.text(1.02, i, f"Holm p={fmt_p(r['p_holm'])}", va="center", fontsize=8,
            transform=ax.get_yaxis_transform())
ax.axvline(0, color="black", ls="--", lw=0.8)
ax.axvspan(-PRACTICAL_DELTA_AUC, PRACTICAL_DELTA_AUC, color="gray", alpha=0.12,
           label=f"±{PRACTICAL_DELTA_AUC:g} practical-equivalence band")
ax.set_yticks(range(len(fp)))
ax.set_yticklabels([f"{MODE_SHORT[[k for k, v in MODE_LABELS.items() if v == r['Mode']][0]]} "
                    f"{'DT' if r['Contrast'].startswith('Deep') else 'TF'} vs Classical"
                    for _, r in fp.iterrows()])
ax.set_xlabel("Estimated difference in ROC-AUC (LME, 95% CI)")
lo = min(fp["ci_low"].min(), -PRACTICAL_DELTA_AUC)
hi = max(fp["ci_high"].max(), PRACTICAL_DELTA_AUC)
pad = 0.08 * (hi - lo)
ax.set_xlim(lo - pad, hi + pad)
ax.set_ylim(-0.7, len(fp) - 0.3)
ax.legend(fontsize=8, loc="lower center", bbox_to_anchor=(0.5, 1.0), frameon=False)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
save_fig(fig, "stats", "Figure11_ForestPlot_LME", dpi=600)
print(fp[["Mode", "Contrast", "estimate", "ci_low", "ci_high", "p_holm"]].round(4).to_string(index=False))

## 5. Ensembles

In [ ]:
# ==========================================================================
# SECTION 5 -- Ensembles
#   (a) composition of each ensemble, read from the results
#   (b) each ensemble vs (i) the best base learner per dataset (oracle,
#       optimistic) and (ii) the best base learner overall in that mode
#       (fixed choice) -- W/T/L, Wilcoxon + Holm, CI, Cliff's delta
#   (c) cost accounting with consistent boundaries:
#       total = everything the ensemble needs; base = standalone cost of its
#       constituents; incremental = total - base (aggregation overhead)
# ==========================================================================
ENS_PROB = ["Soft Voting", "Stacking (sklearn)", "Meta-Stacking"]   # Hard Voting has no ROC-AUC

# ---------- (a) composition ----------
comp_rows = []
for mode in MODE_FOLDERS:
    m = master[master["Mode"] == mode]
    for e in ENSEMBLES:
        if e not in m["Model"].values:
            continue
        pool = CLASSICAL if e != "Meta-Stacking" else BASE_LEARNERS
        keys = m.loc[m["Model"] == e, ["Dataset", "Fold"]].drop_duplicates()
        sizes = [m[(m["Dataset"] == r.Dataset) & (m["Fold"] == r.Fold)
                   & m["Model"].isin(pool)]["Model"].nunique() for r in keys.itertuples()]
        comp_rows.append(dict(Mode=MODE_LABELS[mode], Ensemble=disp(e),
                              Constituents=("7 classical learners" if e != "Meta-Stacking"
                                            else "13 base learners (all families)"),
                              Members_per_fold_median=float(np.median(sizes)) if sizes else np.nan,
                              Members_per_fold_min=min(sizes) if sizes else np.nan,
                              Combiner={"Soft Voting": "mean of predicted probabilities",
                                        "Hard Voting": "majority vote of predicted labels",
                                        "Stacking (sklearn)": "LogisticRegression (sklearn defaults) "
                                                              "on 5-fold CV probabilities "
                                                              "(StackingClassifier, cv=5)",
                                        "Meta-Stacking": f"LogisticRegression(max_iter=5000) on "
                                                         f"{META_INNER_SPLITS}-fold inner-CV OOF "
                                                         "probabilities of the training fold; test "
                                                         "meta-features = mean of the inner-fold "
                                                         "models' test probabilities"}[e]))
comp = pd.DataFrame(comp_rows)
save_table(comp, "ens", "Table_ensemble_composition",
           caption="Composition and combination rule of each ensemble.", label="tab:ens_comp")
print(comp[comp["Mode"] == MODE_LABELS[REFERENCE_MODE]].to_string(index=False))

# ---------- (b) ensemble vs best base ----------
ens_vs_rows, wtl_rows, ens_tables = [], [], {}
for mode in MODE_FOLDERS:
    fm = master[master["Mode"] == mode]
    for metric in ["ROC_AUC", "F1", "MCC"]:
        b = fm[fm["Model"].isin(BASE_LEARNERS)].groupby(["Dataset", "Model"])[metric].mean().unstack()
        e = fm[fm["IsEnsemble"]].groupby(["Dataset", "Model"])[metric].mean().unstack()
        oracle = b.max(axis=1)
        oracle_name = b.idxmax(axis=1)
        overall_best = b.rank(axis=1, ascending=False).mean().idxmin()
        tab = pd.DataFrame({"Best_base_model": oracle_name.map(disp), "Best_base": oracle,
                            f"Best_overall_{disp(overall_best)}": b[overall_best]})
        for en in ENSEMBLES:
            if en in e.columns and e[en].notna().any():
                tab[disp(en)] = e[en]
                for ref_name, ref in [("best base per dataset (oracle)", oracle),
                                      (f"best base overall ({disp(overall_best)})", b[overall_best])]:
                    r = paired_compare(e[en].reindex(ref.index).values, ref.values, tol=1e-6)
                    ens_vs_rows.append(dict(Mode=MODE_LABELS[mode], Metric=metric,
                                            Ensemble=disp(en), Reference=ref_name, **r))
        ens_tables[(mode, metric)] = tab
        if metric == "ROC_AUC":
            save_table((tab.select_dtypes("number") * 100).round(2).join(tab[["Best_base_model"]])
                       .reset_index(), "ens", f"Table6_ensemble_vs_best_base_mode{mode}",
                       caption=f"ROC-AUC (\\%) of the best base learner and the ensembles, "
                               f"{MODE_LABELS[mode]}.", label=f"tab:ens_mode{mode}")
ens_vs = pd.DataFrame(ens_vs_rows)
ens_vs["p_holm"] = np.nan
for (mode, metric), idx in ens_vs.groupby(["Mode", "Metric"]).groups.items():
    ens_vs.loc[idx, "p_holm"] = holm_adjust(ens_vs.loc[idx, "p"].values)
save_table(ens_vs.round(5), "ens", "Table_ensemble_vs_base_tests",
           caption="Ensembles vs best base learner (paired over datasets): wins/ties/losses, "
                   "mean difference with 95\\% bootstrap CI, Wilcoxon signed-rank (Holm-adjusted "
                   "within mode and metric), Cliff's delta.", label="tab:ens_tests")
show = ens_vs[(ens_vs["Mode"] == MODE_LABELS[REFERENCE_MODE]) & (ens_vs["Metric"] == "ROC_AUC")]
print("\n", show[["Ensemble", "Reference", "mean_diff", "ci_low", "ci_high", "wins", "ties",
                  "losses", "p_holm", "cliffs_delta"]].round(4).to_string(index=False))

# ---------- (c) consistent cost accounting ----------
cost_rows = []
for mode in MODE_FOLDERS:
    fm = master[master["Mode"] == mode]
    bsum = {}
    for pool_name, pool in (("classical", CLASSICAL), ("all", BASE_LEARNERS)):
        bsum[pool_name] = (fm[fm["Model"].isin(pool)]
                           .groupby(["Dataset", "Fold"])[["train_time_sec", "infer_time_sec"]].sum())
    for en in ENSEMBLES:
        er = fm[fm["Model"] == en].set_index(["Dataset", "Fold"])[["train_time_sec", "infer_time_sec"]]
        if er.empty:
            continue
        pool = "all" if en == "Meta-Stacking" else "classical"
        bs = bsum[pool].reindex(er.index)
        if en == "Meta-Stacking":
            # Reported train time = inner-CV refits of all base learners + meta fit
            # (no separate full-fold refit is needed). Reported inference time covers only the
            # meta-learner; at deployment the META_INNER_SPLITS inner models of every base
            # learner must also predict -> estimated from the standalone inference times.
            total_train = er["train_time_sec"]
            total_infer = er["infer_time_sec"] + META_INNER_SPLITS * bs["infer_time_sec"]
            incr_train = total_train - bs["train_time_sec"]
            incr_infer = total_infer - bs["infer_time_sec"]
            infer_note = (f"meta-learner (reported) + {META_INNER_SPLITS} x sum of base-learner "
                          "inference times (estimated)")
            train_note = (f"reported: {META_INNER_SPLITS}-fold inner CV of 13 learners + meta fit")
        else:
            total_train = er["train_time_sec"]
            total_infer = er["infer_time_sec"]
            incr_train = total_train - bs["train_time_sec"]
            incr_infer = total_infer - bs["infer_time_sec"]
            infer_note = "reported (constituents predict inside the ensemble)"
            train_note = ("reported: fit of all constituents"
                          + (" + 5-fold internal CV + meta fit" if "Stacking" in en else ""))
        cost_rows.append(dict(
            Mode=MODE_LABELS[mode], Ensemble=disp(en),
            n_learners=7 if pool == "classical" else 13,
            Total_train_s=total_train.mean(), Base_standalone_train_s=bs["train_time_sec"].mean(),
            Incremental_train_s=incr_train.mean(),
            Total_infer_s=total_infer.mean(), Base_standalone_infer_s=bs["infer_time_sec"].mean(),
            Incremental_infer_s=incr_infer.mean(),
            Train_accounting=train_note, Infer_accounting=infer_note))
cost = pd.DataFrame(cost_rows)
save_table(cost.round(4), "ens", "Table18_ensemble_cost_accounting",
           caption="Ensemble cost with consistent boundaries (seconds per outer fold, mean over "
                   "datasets and folds). Total = complete pipeline needed by the ensemble; Base = sum "
                   "of standalone costs of its constituents; Incremental = Total - Base, i.e. "
                   "everything beyond one standalone fit/prediction of each constituent (internal "
                   "CV refits, meta-learner, aggregation). Meta-Stacking total inference is "
                   "estimated from the base learners' standalone inference times.",
           label="tab:ens_cost")
print("\n", cost[cost["Mode"] == MODE_LABELS[REFERENCE_MODE]].drop(columns=["Mode"]).round(3)
      .to_string(index=False))

# ---------- Figure 8 ----------
fm = master[master["Mode"] == REFERENCE_MODE]
top5 = (fm[fm["Model"].isin(BASE_LEARNERS)].groupby("Model")["ROC_AUC"].mean()
        .sort_values(ascending=False).index[:5].tolist())
plot_models = top5 + [e for e in ENSEMBLES if e in fm["Model"].values]
pdf_ = fm[fm["Model"].isin(plot_models)].groupby("Model")[["ROC_AUC", "F1", "MCC"]].mean().reindex(plot_models) * 100
fig, ax = plt.subplots(figsize=(10, 3))
x = np.arange(len(pdf_)); w = 0.25
for i, (m, c) in enumerate(zip(["ROC_AUC", "F1", "MCC"], ["#003f5c", "#ffa600", "#aaaaaa"])):
    ax.bar(x + (i - 1) * w, pdf_[m], width=w, color=c, label=METRIC_DISPLAY[m])
ax.set_xticks(x); ax.set_xticklabels([disp(m) for m in pdf_.index], rotation=15)
ax.set_ylabel("Mean score (%)")
ax.set_ylim(max(0, np.nanmin(pdf_.values) - 10), min(100, np.nanmax(pdf_.values) + 5))
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
ax.legend(ncol=3, fontsize=8)
save_fig(fig, "ens", f"Figure8_ensemble_comparison_mode{REFERENCE_MODE}")

## 6. Computational efficiency

In [ ]:
# ==========================================================================
# SECTION 6a -- Computational efficiency
#   * inference latency normalised per 1,000 predictions (test-fold sizes
#     read from the prediction files) -> comparable across datasets
#   * device: classical learners run on CPU; their recorded "GPU MB" is the
#     memory still allocated by earlier neural models (the pipeline resets the
#     peak counter to the CURRENT allocation, not to zero). Net GPU memory is
#     estimated by subtracting that per-fold baseline.
#   * Pareto front computed programmatically
# ==========================================================================
cm = base[base["Mode"] == REFERENCE_MODE].copy()
cm["n_test"] = [TEST_FOLD_SIZE.get((d, f), np.nan) for d, f in zip(cm["Dataset"], cm["Fold"])]
cm["infer_ms_per_1000"] = cm["infer_time_sec"] / cm["n_test"] * 1000 * 1000
# per-fold GPU baseline = allocation seen by CPU-only (classical) models in that fold
gpu_base = (cm[cm["Model"].isin(CLASSICAL)].groupby(["Dataset", "Fold"])["train_gpu_mb"].min()
            .rename("gpu_baseline_mb"))
cm = cm.join(gpu_base, on=["Dataset", "Fold"])
cm["train_gpu_net_mb"] = (cm["train_gpu_mb"] - cm["gpu_baseline_mb"].fillna(0)).clip(lower=0)
cm.loc[cm["Model"].isin(CLASSICAL), "train_gpu_net_mb"] = 0.0

t7 = cm.groupby("Model").agg(
    Family=("Family", "first"), ROC_AUC=("ROC_AUC", "mean"),
    TrainTime_s=("train_time_sec", "mean"), InferTime_s=("infer_time_sec", "mean"),
    Infer_ms_per_1000=("infer_ms_per_1000", "mean"),
    GPU_raw_max_MB=("train_gpu_mb", "max"), GPU_net_mean_MB=("train_gpu_net_mb", "mean"),
    GPU_net_max_MB=("train_gpu_net_mb", "max")).reindex(BASE_LEARNERS)
t7["ROC_AUC"] *= 100
t7["Device"] = ["CPU" if m in CLASSICAL else "GPU" for m in t7.index]
classical_raw = t7.loc[t7.index.isin(CLASSICAL), "GPU_raw_max_MB"]
if classical_raw.nunique() == 1 and classical_raw.iloc[0] > 0:
    print(f"Classical learners all report the same raw GPU value ({classical_raw.iloc[0]:.2f} MB) "
          "-> confirms it is residual allocation, not model memory.")
t7["Pareto_AUC_vs_train"] = pareto_front(t7, maximize=["ROC_AUC"], minimize=["TrainTime_s"])
t7["Pareto_AUC_vs_infer"] = pareto_front(t7, maximize=["ROC_AUC"], minimize=["Infer_ms_per_1000"])
t7["Pareto_AUC_train_infer"] = pareto_front(t7, maximize=["ROC_AUC"],
                                            minimize=["TrainTime_s", "Infer_ms_per_1000"])
t7.index = t7.index.map(disp)
t7.index.name = "Model"

dominated_by = {}
for m in t7.index:
    dom = [o for o in t7.index if o != m and t7.loc[o, "ROC_AUC"] >= t7.loc[m, "ROC_AUC"]
           and t7.loc[o, "TrainTime_s"] <= t7.loc[m, "TrainTime_s"]
           and (t7.loc[o, "ROC_AUC"] > t7.loc[m, "ROC_AUC"] or t7.loc[o, "TrainTime_s"] < t7.loc[m, "TrainTime_s"])]
    dominated_by[m] = ", ".join(dom)
t7["Dominated_by (AUC vs train)"] = pd.Series(dominated_by)
print(t7.round(3).to_string())


# ---------- Figures ----------
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, xcol, front, xl in [(axes[0], "TrainTime_s", "Pareto_AUC_vs_train", "Mean training time per fold (s, log)"),
                            (axes[1], "Infer_ms_per_1000", "Pareto_AUC_vs_infer", "Inference time per 1,000 predictions (ms, log)")]:
    for fam in FAMILY_ORDER:
        g = t7[t7["Family"] == fam]
        ax.scatter(g[xcol], g["ROC_AUC"], s=60, color=FAMILY_COLORS[fam], label=FAMILY_LABELS[fam],
                   edgecolor="black", linewidth=0.5)
    f = t7[t7[front]].sort_values(xcol)
    ax.step(f[xcol], f["ROC_AUC"], where="post", color="red", lw=1, ls="--", label="Pareto front")
    for m, r in t7.iterrows():
        ax.annotate(m, (r[xcol], r["ROC_AUC"]), xytext=(4, 4), textcoords="offset points", fontsize=8)
    ax.set_xscale("log"); ax.set_xlabel(xl); ax.set_ylabel("Mean ROC-AUC (%)")
    ax.grid(alpha=0.3)
axes[0].legend(fontsize=8)
fig.tight_layout()
save_fig(fig, "comp", f"Figure_Pareto_mode{REFERENCE_MODE}")

fig, ax = plt.subplots(figsize=(7, 6))
for fam in FAMILY_ORDER:
    g = t7[t7["Family"] == fam]
    ax.scatter(g["Infer_ms_per_1000"], g["ROC_AUC"], s=40 + g["GPU_net_max_MB"].fillna(0) * 1.5,
               alpha=0.75, edgecolors="black", linewidth=0.8, label=FAMILY_LABELS[fam],
               color=FAMILY_COLORS[fam])
for m, r in t7.iterrows():
    ax.annotate(m, (r["Infer_ms_per_1000"], r["ROC_AUC"]), xytext=(-10, 8),
                textcoords="offset points", fontsize=9)
ax.set_xscale("log")
ax.set_xlabel("Inference time per 1,000 predictions (ms, log)")
ax.set_ylabel("ROC-AUC (%)")
ax.grid(alpha=0.3); ax.legend(title="Family (bubble = net peak GPU MB)", fontsize=8)
save_fig(fig, "comp", f"Figure_Computational_bubble_mode{REFERENCE_MODE}", dpi=600)

# cost in every mode (compact)
cost_modes = (base.groupby(["Model", "Mode"])[["train_time_sec", "infer_time_sec"]].mean()
              .unstack("Mode"))
cost_modes.columns = [f"{c}_{MODE_SHORT[m]}" for c, m in cost_modes.columns]
cost_modes = cost_modes.reindex(BASE_LEARNERS)
cost_modes.index = cost_modes.index.map(disp)
save_table(cost_modes.round(4).reset_index(names="Model"), "comp", "Table_cost_by_mode",
           caption="Mean training and inference time per fold (s) in each ablation mode.",
           label="tab:cost_modes")

### 6b. Parameter counts (optional, needs pytorch_tabular)

In [ ]:
# ==========================================================================
# SECTION 6b -- Parameter counts of the neural models
#   Builds each network with EXACTLY the pipeline's configuration for the
#   actual number of input features of every dataset (all features for
#   Modes 1-2, the median fold-wise selected subset for Modes 3-4) and
#   counts trainable parameters. No training is performed.
#   Needs pytorch_tabular + pytorch_tabnet (installed below if missing).
# ==========================================================================
PARAMS = None
if COMPUTE_PARAM_COUNTS:
    try:
        import pytorch_tabular, pytorch_tabnet  # noqa: F401
    except ImportError:
        import subprocess, sys
        print("Installing pytorch_tabular / pytorch_tabnet (one-off, ~1-2 min)...")
        subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                        "pytorch_tabular", "pytorch_tabnet"], check=False)
    try:
        import logging, torch
        logging.getLogger("pytorch_tabular").setLevel(logging.ERROR)
        logging.getLogger("lightning").setLevel(logging.ERROR)
        logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
        from pytorch_tabular import TabularModel
        from pytorch_tabular.config import DataConfig, TrainerConfig, OptimizerConfig
        from pytorch_tabular.models.ft_transformer.config import FTTransformerConfig
        from pytorch_tabular.models.tab_transformer.config import TabTransformerConfig
        from pytorch_tabular.models.autoint.config import AutoIntConfig
        from pytorch_tabular.models.gandalf.config import GANDALFConfig
        from pytorch_tabular.models.danet.config import DANetConfig
        from pytorch_tabnet.tab_model import TabNetClassifier

        PT_CFG = {  # identical to pt_model_config_ctors in the pipeline
            "FTTransformer": lambda: FTTransformerConfig(task="classification", learning_rate=1e-3),
            "TabTransformer": lambda: TabTransformerConfig(task="classification", learning_rate=1e-3),
            "AutoInt": lambda: AutoIntConfig(task="classification", learning_rate=1e-3),
            "GANDALF": lambda: GANDALFConfig(task="classification", learning_rate=1e-3),
            "DANet": lambda: DANetConfig(task="classification", learning_rate=1e-3),
        }

        def count_pt(name, n_feat):
            rng = np.random.default_rng(0)
            cols = [f"f{i}" for i in range(n_feat)]
            df = pd.DataFrame(rng.normal(size=(64, n_feat)), columns=cols)
            df["target"] = np.tile([0, 1], 32)
            tm = TabularModel(data_config=DataConfig(target=["target"], continuous_cols=cols),
                              model_config=PT_CFG[name](), optimizer_config=OptimizerConfig(),
                              trainer_config=TrainerConfig(batch_size=256, max_epochs=1,
                                                           accelerator="cpu", progress_bar="none"),
                              verbose=False)
            dm = tm.prepare_dataloader(train=df, validation=df.iloc[:16])
            model = tm.prepare_model(dm)
            return sum(p.numel() for p in model.parameters() if p.requires_grad)

        def count_tabnet(n_feat):
            rng = np.random.default_rng(0)
            X = rng.normal(size=(64, n_feat)).astype(np.float32)
            y = np.tile([0, 1], 32)
            clf = TabNetClassifier(n_d=32, n_a=32, n_steps=5, gamma=1.5, lambda_sparse=1e-4,
                                   optimizer_params=dict(lr=2e-2), mask_type="entmax",
                                   device_name="cpu", verbose=0)
            clf.fit(X, y, max_epochs=1, batch_size=32, virtual_batch_size=16)
            return sum(p.numel() for p in clf.network.parameters() if p.requires_grad)

        feat_all = DATASET_INFO["Features"].dropna().astype(int).to_dict() if "Features" in DATASET_INFO else {}
        feat_sel = {}
        for ds in DATASETS:
            lst = FS_FOLDS.get((4, ds)) or FS_FOLDS.get((3, ds))
            if lst:
                feat_sel[ds] = int(np.median([x["Consensus_Selected"].astype(bool).sum() for x in lst]))
        if not feat_all:
            raise RuntimeError("Number of input features unknown (no feature-selection files).")
        cache, rows = {}, []
        for setting, fmap in (("All features (Modes 1-2)", feat_all),
                              ("Selected features (Modes 3-4)", feat_sel)):
            for ds, nf in fmap.items():
                for mdl in DEEP_TABULAR + TRANSFORMER:
                    key = (mdl, nf)
                    if key not in cache:
                        cache[key] = count_tabnet(nf) if mdl == "TabNet" else count_pt(mdl, nf)
                    rows.append(dict(Setting=setting, Dataset=ds, Model=mdl, n_features=nf,
                                     params=cache[key]))
        PARAMS = pd.DataFrame(rows)
        PARAMS.to_csv(outpath("comp", "Table_param_counts_by_dataset.csv"), index=False)
        psum = PARAMS.groupby(["Model", "Setting"])["params"].agg(["median", "min", "max"]).unstack("Setting")
        psum.columns = [f"{s} [{a}]" for a, s in psum.columns]
        psum = psum.reindex(DEEP_TABULAR + TRANSFORMER)
        psum.index = psum.index.map(disp)
        save_table(psum.reset_index(), "comp", "Table_param_counts_summary", float_format="%.0f",
                   caption="Trainable parameters of the neural models with the pipeline "
                           "configuration (median and range over datasets).",
                   label="tab:params")
        print(psum.to_string())
    except Exception as e:
        print(f"Parameter counting failed ({type(e).__name__}: {e}).\n"
              "Table 7 will show 'n/a' for parameters. Set COMPUTE_PARAM_COUNTS=False to skip.")
        PARAMS = None

# merge parameter counts into Table 7
if PARAMS is not None:
    ref_setting = "Selected" if REFERENCE_MODE in (3, 4) else "All"
    med = (PARAMS[PARAMS["Setting"].str.startswith(ref_setting)]
           .groupby("Model")["params"].median())
    inv_display = {v: k for k, v in MODEL_DISPLAY.items()}
    t7["Parameters_median"] = [med.get(inv_display.get(i, i), np.nan) for i in t7.index]
else:
    t7["Parameters_median"] = np.nan

save_table(t7.round(4).reset_index(), "comp", f"Table7_computational_mode{REFERENCE_MODE}",
           caption=f"Predictive performance and computational cost of the base learners "
                   f"({MODE_LABELS[REFERENCE_MODE]}). Inference latency normalised per 1,000 "
                   "predictions; GPU memory net of the residual allocation measured for CPU-only "
                   "models in the same fold; Pareto flags computed programmatically.",
           label="tab:computational")


def _fmt_params(v):
    if pd.isna(v):
        return "--"
    return f"{v / 1e6:.2f}M" if v >= 1e6 else f"{v / 1e3:.1f}K"


lines = []
for m, r in t7.iterrows():
    lines.append(f"{m:16s} & {FAMILY_LABELS[r['Family']]:12s} & {r['Device']} & {r['ROC_AUC']:.2f} & "
                 f"{r['TrainTime_s']:.2f} & {r['Infer_ms_per_1000']:.2f} & "
                 f"{r['GPU_net_max_MB']:.1f} & {_fmt_params(r['Parameters_median'])} \\\\")
with open(outpath("comp", f"Table7_rows_mode{REFERENCE_MODE}.tex"), "w") as f:
    f.write("% Model & Family & Device & ROC-AUC & Train (s) & Infer (ms/1000) & Net GPU (MB) & Params\n")
    f.write("\n".join(lines))
print("\n".join(lines))

## 7. Export

In [ ]:
# ==========================================================================
# SECTION 7 -- Zip of all outputs
# ==========================================================================
zip_base = os.path.join(os.path.dirname(OUT_ROOT), os.path.basename(OUT_ROOT))
shutil.make_archive(zip_base, "zip", root_dir=OUT_ROOT)
print(f"All outputs: {OUT_ROOT}\nZipped copy: {zip_base}.zip")